# Stage 1.5 -- Validation & Feature Engineering: Panel A (Stock Daily)

## Input
`Data/Data_Collection/Final/Stage_1_Initial_Merge/panel_stock_daily.parquet` (240 columns, ~525K rows, keyed on `(permno, date)`)

## Purpose
Comprehensive validation and feature engineering of the merged stock daily panel before cross-sectional aggregation in Stage 2. The notebook has two major phases: a diagnostics-only validation pass (no data modified), followed by a four-block feature engineering pipeline that transforms raw stock-level columns into stock-comparable ratio features suitable for cross-sectional aggregation.

---

## Phase 1: Validation (Diagnostics Only)

### Check 1: Structural Integrity
- Shape, date range, unique dates, unique PERMNOs
- Primary key `(permno, date)` uniqueness verified via assertion
- Required columns (`permno`, `date`, `dlyret`, `dlycap`) confirmed present
- Dtype summary across all columns
- NaN and zero checks on ID, weight, and target columns

### Check 2: Merge Artefacts & Non-Numeric Columns
- Scans for `_x`/`_y` suffix columns (pandas auto-rename on merge conflicts)
- Identifies non-numeric columns remaining in the factor set (TAQ metadata strings: `symbol`, `cex`, `oex`, `ctime`, `otime`, `ttime_*`, `nbbot_*`)

### Check 3: Exact Duplicate Column Detection
- Samples 10,000 rows and tests all numeric factor pairs for exact value equality (same NaN pattern and identical non-null values)
- Identified `dlyclose` as exact duplicate of `dlyprc`

### Check 4: High-Correlation Pairs (|r| > 0.95)
- Computes pairwise Pearson correlations on a 10,000-row sample across all numeric factors
- Identified 1,874 pairs with |r| > 0.95, falling into three categories: price-level correlations (raw dollar prices all move together), classification-variant redundancy (TAQ tick/WRDS/Lee-Ready variants of the same metric), and scaling correlations (dollar volumes correlated with market cap)
- Near-zero variance factor detection: identifies `c_official` and `o_official` as effectively constant for top-100 S&P 500 stocks

### Check 5: NaN Patterns
- Per-factor NaN tier summary (0%, <5%, 5--15%, 15--30%, >=30%)
- Top 20 highest-NaN factors listed
- All-NaN row count
- NaN co-occurrence between sources (CRSP, TAQ, OptionMetrics) to understand whether missing data clusters by source

### Check 6: Value Range Sanity
- Price columns: non-negative, no zeros
- Return columns: bounded, count of extreme returns (|return| > 50%)
- Volume columns: non-negative, zero counts
- Bid-ask spread: range, negative values, proportion above 10%
- Infinite value scan across all numeric factors

### Check 7: Cross-Panel Alignment
- Panel A vs Panel C (macro daily): trading calendar comparison, identifies dates in one but not the other
- Panel A vs Panel B (stock monthly): PERMNO coverage comparison

### Check 8: Temporal Coverage
- Stocks per day distribution (mean, min, max, std), flags days with fewer than 95 stocks
- Trading days per PERMNO distribution, identifies short-tenure PERMNOs

### Check 9: Full Column Inventory
- Complete listing of all factor columns with dtype, NaN%, min, and max

### Check 10: Summary Verdict
- Aggregates all issues found across checks into a prioritised list (CRITICAL, ACTION, WARNING)

---

## Phase 2: Feature Engineering

### Block 1: Drop Redundant Factors

Six categories of drops, reducing from 240 to 183 columns:

1. **Non-numeric columns (13):** TAQ metadata strings that survived cleaning -- `symbol`, `cex`, `oex`, `ctime`, `otime`, `ttime_1pm`, `ttime_4pm`, `ttime_close`, `ttime_open`, `nbbot_1pm`, `nbbot_4pm`, `nbbot_after_open`, `nbbot_before_close`. Cannot be aggregated numerically.

2. **Exact duplicate (1):** `dlyclose` is identical to `dlyprc` (CRSP stores closing price under both names).

3. **Near-zero variance (2):** `c_official` and `o_official` are binary flags indicating whether the close/open was an official exchange event. For top-100 S&P 500 stocks these are ~99% constant.

4. **Internal identifiers (2):** `tr_seqnum_close` and `tr_seqnum_open` are TAQ internal sequence numbers in the trillions range. Not predictive.

5. **Classification-variant duplicates (30):** TAQ provides the same metric computed under three trade classification algorithms: `_tick` (tick rule), `_wrds` (WRDS proprietary), and `_lr` (Lee-Ready, academic standard). For top-100 large-cap stocks all three produce nearly identical results (r > 0.999). Lee-Ready (`_lr`) is kept as the standard. This applies to: `buyvol`, `sellvol`, `buynumtrades`, `sellnumtrades`, `buy_dv`, `sell_dv`, `total_trade`, `total_dv`, `total_vol`, `avg_buy_price`, `avg_sell_price`, `vwavg_buy_price`, `vwavg_sell_price`.

6. **Institutional size-threshold duplicates (13):** `_inst20k` and `_inst50k` metrics are nearly identical (r ~ 1.0) for large-cap stocks because almost all institutional trades exceed both thresholds. `_inst50k` is kept (more selective, captures true large-block flow), `_inst20k` is dropped. Applies to: `buyvol`, `sellvol`, `buynumtrades`, `sellnumtrades`, `buy_dv`, `sell_dv`, `total_trade`, `total_dv`, `total_vol`, `avg_buy_price`, `avg_sell_price`, `bs_ratio_num`, `bs_ratio_vol`.

### Block 2: Complete Factor Inventory (Pre-Engineering)

Every surviving factor is catalogued with column name, source (CRSP/TAQ/OM), category (price, return, volume, liquidity, order_flow, volatility, etc.), scale type, and description. Scale types determine how each factor is handled in aggregation:

- `ratio` -- already stock-comparable, aggregate directly
- `price_level` -- dollar-denominated, must transform to ratio before aggregating
- `dollar_volume` -- normalise by market cap before aggregating
- `share_volume` -- normalise by shares outstanding before aggregating
- `count` -- normalise by total trades before aggregating
- `time` -- seconds/timing metric
- `exclude` -- do not aggregate (helper column only)

The inventory is saved as `stock_daily_descriptions_pre.csv` for pipeline reference.

### Block 3: Feature Engineering

Four sections executed in order, creating 115 new features and dropping 144 raw columns that are replaced by their derived/normalised versions.

#### Section A: Derived Ratio Features from Price-Level Pairs (22 features)

Transforms raw dollar-denominated price columns into stock-comparable ratios by computing differences and normalising by closing price or midpoint:

- **CRSP intraday features:** `intraday_range` = (high - low) / close, `open_to_close_ret` = (close - open) / open, `turnover` = volume / shares outstanding, `dvol_to_cap` = dollar volume / market cap
- **TAQ buy-sell price premiums:** `buysell_premium_lr`, `buysell_premium_inst50k`, `buysell_premium_retail` -- (avg buy price - avg sell price) / close for each participant type
- **TAQ venue-level ranges:** `venue_range_m/a/b` = (high - low) / close per exchange
- **TAQ intraday drift:** `intraday_drift` = (mid_4pm - mid_open) / mid_open, `midday_drift` = (mid_4pm - mid_1pm) / mid_1pm, `morning_drift` = (mid_1pm - mid_open) / mid_open
- **NBBO spread at different times:** `nbbo_spread_1pm`, `nbbo_spread_4pm`, `nbbo_spread_open`, `nbbo_spread_close` -- (NBO - NBB) / midpoint at each time
- **Close/open positioning:** `close_vs_mid` = (close trade - mid_4pm) / mid_4pm, `open_vs_mid` = (open trade - mid_open) / mid_open
- **VWAP deviation:** `vwap_deviation_m` = (VWAP - close) / close
- **Depth imbalance:** `depth_imbalance` = (bid depth - offer depth) / (bid + offer)

#### Section B: Normalise Volume, Dollar Volume, and Count Columns (69 features)

- **Dollar volumes divided by `dlycap`** (15 columns): `buy_dv_inst50k_to_cap`, `buy_dv_lr_to_cap`, `buy_dv_retail_to_cap`, `sell_dv_inst50k_to_cap`, `sell_dv_lr_to_cap`, `sell_dv_retail_to_cap`, `total_dv_inst50k_to_cap`, `total_dv_lr_to_cap`, `total_dv_retail_to_cap`, `total_dollar_a/b/m_to_cap`, `iso_dollar_to_cap`, `bestbiddepth_dollar_tw_to_cap`, `bestofrdepth_dollar_tw_to_cap`
- **Share volumes divided by `shrout`** (27 columns): buy/sell/total volumes by participant type, venue volumes, ISO volumes, NBBO quantities at 4 times of day, bid/offer depth in shares, trade sizes at open/close/1pm/4pm
- **Trade counts divided by `total_trade`** (13 columns): buy/sell trade counts by participant type, venue trade counts, ISO trade count, outside-NBBO trade count -- all expressed as percentages of total trades

#### Section C: Rolling & Dynamic Features Per Stock (24 features)

All rolling features are computed within each PERMNO using `groupby('permno').transform()`:

- **C1 Return momentum (4):** `ret_cum_5d`, `ret_cum_20d` (cumulative returns), `ret_max_5d`, `ret_min_5d` (extreme returns in window)
- **C2 Return volatility dynamics (3):** `ret_vol_5d`, `ret_vol_20d` (rolling std dev), `ret_vol_ratio` = 5d vol / 20d vol (vol regime detection, >1 means spiking)
- **C3 Volume dynamics (5):** `turnover_5d_mean`, `turnover_20d_mean` (smoothed levels), `turnover_rel_5d`, `turnover_rel_20d` (today vs average, >1 = unusual volume), `dvol_to_cap_rel_5d` (dollar volume intensity surprise) -- note: `dvol_to_cap_rel_5d` was later identified as perfectly redundant with `turnover_rel_5d` (r=1.0) and dropped
- **C4 Liquidity dynamics (5):** `spread_5d_mean`, `spread_20d_mean`, `spread_rel_5d`, `spread_rel_20d` (today's spread / recent average, >1 = deteriorating liquidity), `effspread_pct_rel_5d` (TAQ effective spread surprise)
- **C5 Intraday pattern dynamics (3):** `intraday_range_rel_5d` (range breakout signal), `intraday_range_5d_mean`, `open_to_close_ret_5d_mean` (sustained intraday direction)
- **C6 Implied volatility dynamics (6):** `iv_catm_chg_1d`, `iv_catm_chg_5d` (IV momentum), `iv_catm_rel_20d` (IV regime), `vrp_rv_chg_5d` (VRP shift), `skew_chg_5d` (tail fear shift), `vol_term_chg_5d` (term structure dynamics)
- **C7 Order flow dynamics (5):** `bs_ratio_vol_5d_mean`, `bs_ratio_vol_chg_1d` (flow persistence and reversal), `bs_ratio_inst_5d_mean`, `bs_ratio_inst_chg_5d` (institutional flow dynamics), `retail_vol_5d_mean` (retail participation trend)
- **C8 Options positioning dynamics (4):** `pc_ratio_chg_5d`, `pc_ratio_rel_20d` (sentiment shift), `gex_norm_chg_5d` (dealer hedging pressure change), `total_oi_norm_chg_5d` (options market growth)
- **C9 Price impact dynamics (2):** `priceimpact_rel_5d`, `realspread_rel_5d` (today vs 5d average)
- **C10 Depth dynamics (4):** `depth_imbalance_5d_mean`, `depth_imbalance_chg_1d`, `bid_depth_rel_5d`, `offer_depth_rel_5d` (depth withdrawal signals)
- **C11 Cross-metric interactions (1):** `volume_return_corr_20d` -- rolling 20-day correlation between return and turnover. Positive = high volume on up days (conviction), negative = selling pressure.

#### Section D: Drop Original Raw Columns (144 columns dropped)

All columns that were replaced by derived or normalised versions are dropped:
- All price-level columns (42): CRSP prices, TAQ average prices, midpoints, NBBO, pre-trade midpoints, close/open prices, VWAP, price extremes
- All original dollar volume columns (15)
- All original share volume columns (27)
- All original trade count columns (13)
- Helper columns: `dlyprcvol`, `dlyvol` (replaced by `dvol_to_cap` and `turnover`), `shrout` (normalisation denominator only), `total_trade` (count normalisation denominator), `stime_close`, `stime_open` (timing in seconds, not useful cross-sectionally)

#### Post-Block 3 Adjustments
- `dvol_to_cap_rel_5d` dropped (identified as perfectly redundant with `turnover_rel_5d`, r=1.0)
- `retail_dv_share` added: retail dollar volume as percentage of total Lee-Ready dollar volume, computed as `(buy_dv_retail_to_cap + sell_dv_retail_to_cap) / total_dv_lr_to_cap`

### Block 4: Final Factor Inventory & Save

Every surviving factor is catalogued in a final inventory DataFrame with column name, source, category, and description. The inventory is validated against the actual data columns (bidirectional check: every factor in data is catalogued, every catalogued factor exists in data). Summary statistics by source and category are printed.

**Final factor count: 196** (from 240 original columns: 57 dropped in Block 1, 115 new features created in Block 3, 144 raw columns replaced in Block 3, plus post-adjustments)

**Factor breakdown by source:**
- CRSP: ~24 factors (returns, liquidity, volume, momentum, volatility dynamics)
- TAQ: ~131 factors (spreads, price impact, order flow ratios, normalised volumes/counts, depth, venue metrics, microstructure quality, plus all derived ratios and rolling dynamics)
- OM: ~40 factors (implied volatility surface, realised vol, VRP, put-call metrics, Greeks/positioning, moneyness distribution, plus IV/positioning dynamics)
- CRSP+TAQ: 1 factor (volume-return correlation)

**Key property after engineering:** every factor is stock-comparable (ratio, percentage, or already-normalised metric). No raw dollar-denominated or share-denominated columns remain. The panel is ready for cross-sectional aggregation in Stage 2.

## Outputs
- `Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering/panel_stock_daily_engineered.parquet` -- 196 factor columns plus `permno`, `date`, `dlyret` (target), `dlycap` (weight)
- `Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering/stock_daily_descriptions_pre.csv` -- pre-engineering factor inventory (183 factors)
- `Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering/stock_daily_factor_inventory_final.csv` -- final post-engineering factor inventory (196 factors with source, category, description)

In [2]:
# %% [markdown]
# # Stage 1.5 — Validation: Panel A (Stock Daily)
#
# Comprehensive validation of panel_stock_daily.parquet before feature
# engineering and aggregation. No data is modified — diagnostics only.
#
# Checks:
#   1. Structural integrity (shape, keys, dtypes)
#   2. Merge artefact detection + non-numeric column detection
#   3. Exact duplicate column detection
#   4. High-correlation pairs (>0.95) — potential redundancy
#   5. NaN patterns
#   6. Value range sanity
#   7. Cross-panel alignment
#   8. Temporal coverage
#   9. Full column inventory
#  10. Summary verdict

# %%
import pandas as pd
import numpy as np
from pathlib import Path
from itertools import combinations

PANEL_A_PATH = Path('../../../Data/Data_Collection/Final/Stage_1_Initial_Merge/panel_stock_daily.parquet')
PANEL_C_PATH = Path('../../../Data/Data_Collection/Final/Stage_1_Initial_Merge/panel_macro_daily.parquet')
PANEL_B_PATH = Path('../../../Data/Data_Collection/Final/Stage_1_Initial_Merge/panel_stock_monthly.parquet')

# ═══════════════════════════════════════════════════════════════════════════════
# LOAD DATA AND DEFINE COLUMN GROUPS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
df = pd.read_parquet(PANEL_A_PATH)
df['date'] = pd.to_datetime(df['date'])

# Define column groups once, use everywhere
id_cols = ['permno', 'date']
weight_cols = ['dlycap']
target_cols = ['dlyret']
meta_cols = id_cols + weight_cols + target_cols

factor_cols = [c for c in df.columns if c not in meta_cols]
numeric_factors = [c for c in factor_cols if pd.api.types.is_numeric_dtype(df[c])]
non_numeric_factors = [c for c in factor_cols if c not in numeric_factors]

print(f"Loaded Panel A: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Factor columns: {len(factor_cols)} total")
print(f"  Numeric factors: {len(numeric_factors)}")
print(f"  Non-numeric factors: {len(non_numeric_factors)}")

# ═══════════════════════════════════════════════════════════════════════════════
# CHECK 1: STRUCTURAL INTEGRITY
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("CHECK 1: STRUCTURAL INTEGRITY")
print("=" * 90)

print(f"\n  Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"  Unique dates: {df['date'].nunique():,}")
print(f"  Unique PERMNOs: {df['permno'].nunique()}")

# Key uniqueness
n_dupes = df.duplicated(subset=['permno', 'date']).sum()
print(f"\n  Duplicate (permno, date): {n_dupes}")
assert n_dupes == 0, f"FAIL: {n_dupes} duplicate keys!"
print(f"  ✓ Primary key (permno, date) is unique")

# Expected columns present
required = ['permno', 'date', 'dlyret', 'dlycap']
missing_required = [c for c in required if c not in df.columns]
if missing_required:
    print(f"\n  ✗ MISSING required columns: {missing_required}")
else:
    print(f"  ✓ All required columns present (permno, date, dlyret, dlycap)")

# Dtype summary
print(f"\n  Dtype summary:")
dtype_counts = df.dtypes.value_counts()
for dtype, count in dtype_counts.items():
    print(f"    {str(dtype):<20s} {count:>4d} columns")

# ID/weight/target integrity
print(f"\n  ID column integrity:")
print(f"    permno NaN: {df['permno'].isna().sum()}")
print(f"    date NaN:   {df['date'].isna().sum()}")
print(f"    dlyret NaN: {df['dlyret'].isna().sum()}")
print(f"    dlycap NaN: {df['dlycap'].isna().sum()}")
print(f"    dlycap zero: {(df['dlycap'] == 0).sum()}")

# ═══════════════════════════════════════════════════════════════════════════════
# CHECK 2: MERGE ARTEFACTS & NON-NUMERIC COLUMNS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("CHECK 2: MERGE ARTEFACTS & NON-NUMERIC COLUMNS")
print("=" * 90)

# Check for _x, _y suffix columns (pandas auto-rename on conflicts)
suffix_cols = [c for c in df.columns if c.endswith('_x') or c.endswith('_y')]
if suffix_cols:
    print(f"\n  ✗ MERGE ARTEFACT: Found {len(suffix_cols)} columns with _x/_y suffixes:")
    for c in suffix_cols:
        print(f"    {c}")
else:
    print(f"\n  ✓ No _x/_y suffix columns (merge was clean)")

# Non-numeric columns in factor set
if non_numeric_factors:
    print(f"\n  ⚠ Non-numeric columns in factor set ({len(non_numeric_factors)}):")
    for c in non_numeric_factors:
        unique_vals = df[c].dropna().unique()[:5]
        print(f"    {c:<30s} dtype={str(df[c].dtype):<10s} samples: {unique_vals}")
    print(f"  → These should be dropped in feature engineering")
else:
    print(f"  ✓ All factor columns are numeric")

# ═══════════════════════════════════════════════════════════════════════════════
# CHECK 3: EXACT DUPLICATE COLUMN DETECTION
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("CHECK 3: EXACT DUPLICATE COLUMN DETECTION")
print("=" * 90)

# Sample 10,000 rows for speed
sample_size = min(10_000, len(df))
sample_idx = df.sample(sample_size, random_state=42).index

exact_dupes = []
cols_checked = set()

for i, c1 in enumerate(numeric_factors):
    if c1 in cols_checked:
        continue
    for c2 in numeric_factors[i+1:]:
        if c2 in cols_checked:
            continue
        s1 = df.loc[sample_idx, c1]
        s2 = df.loc[sample_idx, c2]
        # Quick check: same NaN pattern
        if s1.isna().equals(s2.isna()):
            valid = pd.DataFrame({'a': s1, 'b': s2}).dropna()
            if len(valid) > 0 and valid['a'].equals(valid['b']):
                exact_dupes.append((c1, c2))
                cols_checked.add(c2)

if exact_dupes:
    print(f"\n  ✗ Found {len(exact_dupes)} exact duplicate column pairs:")
    for c1, c2 in exact_dupes:
        print(f"    {c1} == {c2}")
else:
    print(f"\n  ✓ No exact duplicate columns found (checked {len(numeric_factors)} numeric factors)")

# ═══════════════════════════════════════════════════════════════════════════════
# CHECK 4: HIGH-CORRELATION PAIRS (>0.95)
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("CHECK 4: HIGH-CORRELATION PAIRS (>0.95)")
print("=" * 90)

sample = df.loc[sample_idx, numeric_factors]

print(f"\n  Computing correlations on {len(sample):,} sampled rows × "
      f"{len(numeric_factors)} numeric factors...")
corr = sample.corr(method='pearson', min_periods=100)

# Find pairs with |correlation| > 0.95
high_corr_pairs = []
for i, c1 in enumerate(numeric_factors):
    for c2 in numeric_factors[i+1:]:
        if c1 in corr.columns and c2 in corr.columns:
            r = corr.loc[c1, c2]
            if pd.notna(r) and abs(r) > 0.95:
                high_corr_pairs.append((c1, c2, r))

high_corr_pairs.sort(key=lambda x: abs(x[2]), reverse=True)

if high_corr_pairs:
    print(f"\n  Found {len(high_corr_pairs)} pairs with |r| > 0.95:")
    print(f"\n  {'Col 1':<35s} {'Col 2':<35s} {'Corr':>8s}")
    print("  " + "-" * 80)
    for c1, c2, r in high_corr_pairs[:30]:
        print(f"  {c1:<35s} {c2:<35s} {r:>+8.4f}")
    if len(high_corr_pairs) > 30:
        print(f"  ... and {len(high_corr_pairs) - 30} more pairs")
else:
    print(f"\n  ✓ No factor pairs with |r| > 0.95")

# Near-zero variance factors
print(f"\n--- Near-zero variance factors ---")
low_var = []
for c in numeric_factors:
    vals = sample[c].dropna()
    if len(vals) > 100:
        if vals.std() < 1e-10 or vals.nunique() <= 3:
            low_var.append((c, vals.nunique(), vals.std()))

if low_var:
    print(f"  Found {len(low_var)} near-zero variance factors:")
    for c, nunique, std in low_var:
        print(f"    {c:<35s} {nunique:>4d} unique values, std={std:.2e}")
else:
    print(f"  ✓ No near-zero variance factors")

# ═══════════════════════════════════════════════════════════════════════════════
# CHECK 5: NaN PATTERNS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("CHECK 5: NaN PATTERNS")
print("=" * 90)

nan_pct = (df[factor_cols].isna().mean() * 100).round(2)
nan_sorted = nan_pct.sort_values(ascending=False)

# Tier summary
t0 = (nan_pct == 0).sum()
t1 = ((nan_pct > 0) & (nan_pct < 5)).sum()
t2 = ((nan_pct >= 5) & (nan_pct < 15)).sum()
t3 = ((nan_pct >= 15) & (nan_pct < 30)).sum()
t4 = (nan_pct >= 30).sum()

print(f"\n  NaN tiers across {len(factor_cols)} factors:")
print(f"    0% NaN:       {t0}")
print(f"    <5% NaN:      {t1}")
print(f"    5-15% NaN:    {t2}")
print(f"    15-30% NaN:   {t3}")
print(f"    ≥30% NaN:     {t4}  {'← INVESTIGATE' if t4 > 0 else ''}")

# Show worst 20
if nan_sorted.iloc[0] > 0:
    print(f"\n  Top 20 highest NaN factors:")
    print(f"  {'Factor':<35s} {'NaN %':>8s}")
    print("  " + "-" * 45)
    for col in nan_sorted.head(20).index:
        print(f"  {col:<35s} {nan_sorted[col]:>7.2f}%")

# All-NaN rows
all_nan_rows = df[factor_cols].isna().all(axis=1).sum()
print(f"\n  Rows where ALL factors are NaN: {all_nan_rows}")

# NaN co-occurrence between sources
print(f"\n--- NaN co-occurrence between sources ---")
source_reps = {}
crsp_candidates = [c for c in ['dlyprc', 'dlyretx', 'dlyvol'] if c in df.columns]
if crsp_candidates:
    source_reps['CRSP'] = crsp_candidates[0]

taq_candidates = [c for c in numeric_factors
                  if c.startswith(('avg_', 'total_', 'vw_', 'n_', 'mid_'))]
if taq_candidates:
    source_reps['TAQ'] = taq_candidates[0]

om_candidates = [c for c in numeric_factors
                 if c.startswith(('iv_', 'Skew_', 'PC_', 'hvol', 'vrp_', 'gex_', 'oi_wt_'))]
if om_candidates:
    source_reps['OM'] = om_candidates[0]

if len(source_reps) >= 2:
    for (s1, c1), (s2, c2) in combinations(source_reps.items(), 2):
        both_nan = (df[c1].isna() & df[c2].isna()).sum()
        either_nan = (df[c1].isna() | df[c2].isna()).sum()
        print(f"  {s1} ({c1}) & {s2} ({c2}):")
        print(f"    Both NaN: {both_nan:,}  Either NaN: {either_nan:,}")

# ═══════════════════════════════════════════════════════════════════════════════
# CHECK 6: VALUE RANGE SANITY
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("CHECK 6: VALUE RANGE SANITY")
print("=" * 90)

# Price columns: non-negative
print(f"\n--- Price column checks ---")
price_cols = [c for c in ['dlyprc', 'dlyopen', 'dlyhigh', 'dlylow', 'dlyclose',
                           'dlybid', 'dlyask'] if c in df.columns]
for c in price_cols:
    vals = df[c].dropna()
    n_neg = (vals < 0).sum()
    n_zero = (vals == 0).sum()
    print(f"  {c:<15s} range: [{vals.min():.2f}, {vals.max():.2f}]  "
          f"neg: {n_neg}  zero: {n_zero}")

# Return columns: bounded
print(f"\n--- Return column checks ---")
ret_cols = [c for c in ['dlyret', 'dlyretx', 'dlyreti'] if c in df.columns]
for c in ret_cols:
    vals = df[c].dropna()
    n_extreme = (vals.abs() > 0.5).sum()
    print(f"  {c:<15s} range: [{vals.min():.4f}, {vals.max():.4f}]  "
          f"|ret|>50%: {n_extreme}")

# Volume columns: non-negative
print(f"\n--- Volume column checks ---")
vol_cols = [c for c in ['dlyvol', 'dlyprcvol'] if c in df.columns]
for c in vol_cols:
    vals = df[c].dropna()
    n_neg = (vals < 0).sum()
    n_zero = (vals == 0).sum()
    print(f"  {c:<15s} range: [{vals.min():,.0f}, {vals.max():,.0f}]  "
          f"neg: {n_neg}  zero: {n_zero}")

# Bid-ask spread
if 'bid_ask_spread' in df.columns:
    print(f"\n--- Bid-ask spread check ---")
    bas = df['bid_ask_spread'].dropna()
    print(f"  Range: [{bas.min():.6f}, {bas.max():.6f}]")
    print(f"  Median: {bas.median():.6f}")
    print(f"  Negative: {(bas < 0).sum()}")
    print(f"  > 10%: {(bas > 0.10).sum()} ({(bas > 0.10).mean()*100:.2f}%)")

# Infinite values
print(f"\n--- Infinite value check ---")
n_inf = 0
inf_cols = []
for c in numeric_factors:
    try:
        arr = df[c].to_numpy(dtype='float64', na_value=0)
        n = np.isinf(arr).sum()
        if n > 0:
            inf_cols.append((c, n))
            n_inf += n
    except (ValueError, TypeError):
        pass

if inf_cols:
    print(f"  ✗ Found {n_inf} infinite values in {len(inf_cols)} columns:")
    for c, n in inf_cols:
        print(f"    {c:<35s} {n:>6d}")
else:
    print(f"  ✓ No infinite values")

# ═══════════════════════════════════════════════════════════════════════════════
# CHECK 7: CROSS-PANEL ALIGNMENT
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("CHECK 7: CROSS-PANEL ALIGNMENT")
print("=" * 90)

# Panel A vs Panel C (trading calendar)
print(f"\n--- Panel A vs Panel C (trading calendar) ---")
panel_c = pd.read_parquet(PANEL_C_PATH, columns=['date'])
panel_c['date'] = pd.to_datetime(panel_c['date'])
c_dates = set(panel_c['date'])
a_dates = set(df['date'].unique())

in_a_not_c = a_dates - c_dates
in_c_not_a = c_dates - a_dates

print(f"  Panel A unique dates: {len(a_dates):,}")
print(f"  Panel C unique dates: {len(c_dates):,}")
print(f"  In A but not C: {len(in_a_not_c)}")
print(f"  In C but not A: {len(in_c_not_a)}")

if len(in_a_not_c) == 0 and len(in_c_not_a) == 0:
    print(f"  ✓ Trading calendars match perfectly")
else:
    if in_a_not_c:
        print(f"  Dates in A not in C: {sorted(in_a_not_c)[:5]}")
    if in_c_not_a:
        print(f"  Dates in C not in A: {sorted(in_c_not_a)[:5]}")

del panel_c

# Panel A vs Panel B (PERMNO coverage)
print(f"\n--- Panel A vs Panel B (PERMNO coverage) ---")
panel_b = pd.read_parquet(PANEL_B_PATH, columns=['permno', 'date'])
panel_b['date'] = pd.to_datetime(panel_b['date'])

a_permnos = set(df['permno'].unique())
b_permnos = set(panel_b['permno'].unique())

in_a_not_b = a_permnos - b_permnos
in_b_not_a = b_permnos - a_permnos

print(f"  Panel A PERMNOs: {len(a_permnos)}")
print(f"  Panel B PERMNOs: {len(b_permnos)}")
print(f"  In A but not B: {len(in_a_not_b)} "
      f"{sorted([int(p) for p in in_a_not_b]) if in_a_not_b else ''}")
print(f"  In B but not A: {len(in_b_not_a)} "
      f"{sorted([int(p) for p in in_b_not_a]) if in_b_not_a else ''}")

if a_permnos == b_permnos:
    print(f"  ✓ PERMNOs match perfectly between Panel A and Panel B")

del panel_b

# ═══════════════════════════════════════════════════════════════════════════════
# CHECK 8: TEMPORAL COVERAGE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("CHECK 8: TEMPORAL COVERAGE")
print("=" * 90)

# Stocks per day
stocks_per_day = df.groupby('date')['permno'].nunique()
print(f"\n  Stocks per day:")
print(f"    Mean: {stocks_per_day.mean():.1f}")
print(f"    Min:  {stocks_per_day.min()} (on {stocks_per_day.idxmin().date()})")
print(f"    Max:  {stocks_per_day.max()} (on {stocks_per_day.idxmax().date()})")
print(f"    Std:  {stocks_per_day.std():.2f}")

thin_days = stocks_per_day[stocks_per_day < 95]
if len(thin_days) > 0:
    print(f"\n  Days with <95 stocks: {len(thin_days)}")
    for date, n in thin_days.head(10).items():
        print(f"    {date.date()}: {n} stocks")
else:
    print(f"\n  ✓ All days have ≥95 stocks")

# Per-PERMNO coverage
permno_days = df.groupby('permno')['date'].nunique()
print(f"\n  Trading days per PERMNO:")
print(f"    Mean: {permno_days.mean():.0f}")
print(f"    Min:  {permno_days.min()} (PERMNO {permno_days.idxmin()})")
print(f"    Max:  {permno_days.max()} (PERMNO {permno_days.idxmax()})")

short_permnos = permno_days[permno_days < 252]
print(f"  PERMNOs with <252 trading days: {len(short_permnos)}")

# ═══════════════════════════════════════════════════════════════════════════════
# CHECK 9: FULL COLUMN INVENTORY
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("CHECK 9: FULL COLUMN INVENTORY")
print("=" * 90)

print(f"\n  Total columns: {len(df.columns)}")
print(f"  ID + weight + target: {len(meta_cols)} — {meta_cols}")
print(f"  Factor columns: {len(factor_cols)} ({len(numeric_factors)} numeric, "
      f"{len(non_numeric_factors)} non-numeric)")

print(f"\n  Complete factor list ({len(factor_cols)} columns):")
print(f"  {'#':<5s} {'Factor':<40s} {'Dtype':<12s} {'NaN%':>7s}  {'Min':>12s}  {'Max':>12s}")
print("  " + "-" * 85)

for i, c in enumerate(factor_cols, 1):
    dtype = str(df[c].dtype)
    nan_p = df[c].isna().mean() * 100
    if pd.api.types.is_numeric_dtype(df[c]):
        vals = df[c].dropna()
        if len(vals) > 0:
            mn = f"{vals.min():.4f}"
            mx = f"{vals.max():.4f}"
        else:
            mn = mx = "N/A"
    else:
        mn = mx = "non-numeric"
    print(f"  {i:<5d} {c:<40s} {dtype:<12s} {nan_p:>6.2f}%  {mn:>12s}  {mx:>12s}")

# ═══════════════════════════════════════════════════════════════════════════════
# CHECK 10: SUMMARY VERDICT
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("CHECK 10: SUMMARY VERDICT")
print("=" * 90)

issues = []

if n_dupes > 0:
    issues.append(f"CRITICAL: {n_dupes} duplicate (permno, date) keys")
if suffix_cols:
    issues.append(f"CRITICAL: {len(suffix_cols)} merge artefact columns ({suffix_cols})")
if non_numeric_factors:
    issues.append(f"ACTION: {len(non_numeric_factors)} non-numeric factor columns to drop: {non_numeric_factors}")
if exact_dupes:
    issues.append(f"ACTION: {len(exact_dupes)} exact duplicate column pairs to resolve")
if t4 > 0:
    issues.append(f"WARNING: {t4} factors with ≥30% NaN")
if inf_cols:
    issues.append(f"WARNING: Infinite values in {len(inf_cols)} columns")
if all_nan_rows > 0:
    issues.append(f"WARNING: {all_nan_rows} rows with all-NaN factors")
if in_a_not_c or in_c_not_a:
    issues.append(f"WARNING: Trading calendar mismatch with Panel C")

if issues:
    print(f"\n  Issues found ({len(issues)}):")
    for issue in issues:
        print(f"    • {issue}")
else:
    print(f"\n  ✓ ALL CHECKS PASSED")

print(f"""
  Panel A Summary:
    Rows:                  {len(df):,}
    Total factors:         {len(factor_cols)}
    Numeric factors:       {len(numeric_factors)}
    Non-numeric factors:   {len(non_numeric_factors)}
    Date range:            {df['date'].min().date()} → {df['date'].max().date()}
    PERMNOs:               {df['permno'].nunique()}
    Stocks/day:            {stocks_per_day.mean():.1f} avg
    Key integrity:         ✓ unique (permno, date)
    High-corr pairs:       {len(high_corr_pairs)} (|r| > 0.95)
    Merge artefacts:       {len(suffix_cols)}
    Exact duplicate cols:  {len(exact_dupes)}
""")

print("Validation complete. Review high-correlation pairs, non-numeric columns,")
print("and full column inventory before proceeding to feature engineering.")

Loaded Panel A: 525,957 rows × 240 columns
  Factor columns: 236 total
  Numeric factors: 223
  Non-numeric factors: 13
CHECK 1: STRUCTURAL INTEGRITY

  Shape: 525,957 rows × 240 columns
  Date range: 2004-01-02 → 2024-12-31
  Unique dates: 5,285
  Unique PERMNOs: 227

  Duplicate (permno, date): 0
  ✓ Primary key (permno, date) is unique
  ✓ All required columns present (permno, date, dlyret, dlycap)

  Dtype summary:
    Float64               169 columns
    Int64                  44 columns
    string                 13 columns
    float64                13 columns
    datetime64[ns]          1 columns

  ID column integrity:
    permno NaN: 0
    date NaN:   0
    dlyret NaN: 0
    dlycap NaN: 0
    dlycap zero: 0

CHECK 2: MERGE ARTEFACTS & NON-NUMERIC COLUMNS

  ✓ No _x/_y suffix columns (merge was clean)

  ⚠ Non-numeric columns in factor set (13):
    cex                            dtype=string     samples: <StringArray>
['C', 'Q', 'A', 'N', 'P']
Length: 5, dtype: string
    ct

## Stage 4: Clean & Save (Feature Engineering Pre-processing)

### Column Redundancy & Metadata Cleanup
**13 non-numeric columns dropped:** 
These are TAQ metadata columns that are categorical or string-based and not predictive factors:
*   `symbol`: Redundant (using PERMNO).
*   `cex`, `oex`: Close/open exchange codes.
*   `ctime`, `otime`: Timestamps as strings.
*   `ttime_*`, `nbbot_*`: Remaining TAQ timestamp strings that survived previous cleaning rounds.

**Exact duplicate identified:** 
`dlyclose` is an exact duplicate of `dlyprc`. Dropped `dlyclose` in favor of `dlyprc` (the CRSP standard).

### Correlation & Variance Analysis
**1,874 high-correlation pairs identified:** 
Analysis shows these fall into three manageable categories:
*   **Price-Level Correlations:** `dlyprc`, `dlybid`, `dlyask`, `cprc`, etc., correlate at ~1.0 because they measure raw dollar prices. These are retained for feature computation (ratios) but **excluded from final aggregation** to prevent scale bias.
*   **Classification-Variant Redundancy:** TAQ provides metrics computed via multiple methods (Tick vs. WRDS vs. Lee-Ready). For each triplet, we keep the **Lee-Ready (_lr)** variant as the academic standard and drop the others (e.g., `_tick`, `_wrds`).
*   **Scaling Correlations:** Dollar-volume and size metrics (e.g., `buy_dv_lr`) correlate with market cap. These will be decorrelated in the feature engineering step by normalizing by `dlycap`.

**Near-zero variance factors:** 
`c_official` and `o_official` (binary flags for "official" exchange prices) are effectively constant for S&P 100 stocks. Both are dropped.

### Data Integrity & Value Ranges
*   **NaN Tiers:** No factors exceed the 30% threshold. The 15–22% NaN ranges are expected for late-starting series (BATS, ISO, and retail factors).
*   **Outliers:** Extreme max values in `oprc` ($100,000) and `price_high_a` ($150,500) likely represent Berkshire Hathaway (BRK.A) or data artifacts. Because raw prices are excluded from aggregation and outliers like `dollarrealizedspread_lr_dw` will be winsorized, no manual row deletion is required.
*   **Non-Factors:** `tr_seqnum_close` and `tr_seqnum_open` (internal TAQ identifiers in the trillions) are dropped.

---

### Summary: Actions for Feature Engineering Notebook

| Action | Target Columns |
| :--- | :--- |
| **Drop Strings** | 13 columns including `symbol`, `ctime`, `ttime_*`, `nbbot_*` |
| **Drop Duplicates** | `dlyclose`, `tr_seqnum_close/open`, `c_official`, `o_official` |
| **Consolidate** | Remove `_tick` and `_wrds` variants; keep `_lr` (Lee-Ready) |
| **Exclude** | Flag raw price columns (`dlyprc`, `dlybid`, `mid_*`, etc.) for exclusion after ratio computation |
| **Alignment** | Proceed with merge (calendars and PERMNOs show 100% alignment) |

In [3]:
# %% [markdown]
# # Stage 1.5 — Feature Engineering: Panel A (Stock Daily)
#
# Block 1: Drop redundant, non-numeric, and uninformative factors
# Block 2: Inventory of surviving factors (TBD)
# Block 3: Feature engineering — derived factors (TBD)
# Block 4: Final factor inventory for aggregation (TBD)
#
# Input:  Data/Data_Collection/Final/Stage_1_Initial_Merge/panel_stock_daily.parquet
# Output: Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering/panel_stock_daily_engineered.parquet

# %%
import pandas as pd
import numpy as np
from pathlib import Path

IN_PATH  = Path('../../../Data/Data_Collection/Final/Stage_1_Initial_Merge/panel_stock_daily.parquet')
OUT_DIR  = Path('../../../Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering')
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(IN_PATH)
df['date'] = pd.to_datetime(df['date'])

print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
n_start = df.shape[1]

# ═══════════════════════════════════════════════════════════════════════════════
# BLOCK 1: DROP REDUNDANT FACTORS
# ═══════════════════════════════════════════════════════════════════════════════

# %% [markdown]
# ## Block 1: Drop Redundant Factors
#
# Six categories of drops, each with full reasoning:
#
# 1. **Non-numeric columns** — string timestamps, exchange codes, ticker symbol.
#    These are TAQ metadata that survived the cleaning stage. Cannot be
#    aggregated numerically.
#
# 2. **Exact duplicate** — `dlyclose` is identical to `dlyprc` (CRSP stores
#    closing price under both names).
#
# 3. **Near-zero variance** — `c_official` and `o_official` are binary flags
#    indicating whether the close/open was an official exchange event. For
#    top-100 S&P 500 stocks these are ~99% constant (always official).
#
# 4. **Internal identifiers** — `tr_seqnum_close` and `tr_seqnum_open` are
#    TAQ internal sequence numbers (in the trillions). Not predictive factors.
#
# 5. **Classification-variant duplicates** — TAQ provides the same metric
#    computed under different trade classification algorithms:
#      - `_tick` = tick rule
#      - `_wrds` = WRDS proprietary classification
#      - `_lr` = Lee-Ready algorithm (academic standard)
#    For top-100 large-cap stocks, these produce nearly identical results
#    (r > 0.999). Keep `_lr` only as the academic standard. This applies to:
#    buyvol, sellvol, buynumtrades, sellnumtrades, buy_dv, sell_dv,
#    total_trade, total_dv, avg_buy_price, avg_sell_price, vwavg_buy_price,
#    vwavg_sell_price.
#
# 6. **Institutional size-threshold duplicates** — `_inst20k` and `_inst50k`
#    metrics are nearly identical (r ~ 1.0) for large-cap stocks because
#    almost all institutional trades exceed both thresholds. Keep `_inst50k`
#    (more selective, captures true large-block flow), drop `_inst20k`.

# %%
print("=" * 90)
print("BLOCK 1: DROP REDUNDANT FACTORS")
print("=" * 90)

# ── 1. Non-numeric columns (TAQ metadata) ───────────────────────────────────
drop_non_numeric = [
    'symbol',           # Ticker symbol — redundant, we use PERMNO
    'cex',              # Close exchange code (string: C, Q, A, N, P)
    'oex',              # Open exchange code (string: P, D, Q, C, B)
    'ctime',            # Close timestamp as string
    'otime',            # Open timestamp as string
    'ttime_1pm',        # Trade timestamp at 1pm (string)
    'ttime_4pm',        # Trade timestamp at 4pm (string)
    'ttime_close',      # Trade timestamp at close (string)
    'ttime_open',       # Trade timestamp at open (string)
    'nbbot_1pm',        # NBBO timestamp at 1pm (string)
    'nbbot_4pm',        # NBBO timestamp at 4pm (string)
    'nbbot_after_open',   # NBBO timestamp after open (string)
    'nbbot_before_close', # NBBO timestamp before close (string)
]

# ── 2. Exact duplicate ──────────────────────────────────────────────────────
drop_exact_dupe = [
    'dlyclose',         # Identical to dlyprc (CRSP closing price under two names)
]

# ── 3. Near-zero variance ───────────────────────────────────────────────────
drop_low_var = [
    'c_official',       # Binary: was close official? ~99% = 1 for S&P 100
    'o_official',       # Binary: was open official? ~99% = 1 for S&P 100
]

# ── 4. Internal identifiers (not factors) ───────────────────────────────────
drop_identifiers = [
    'tr_seqnum_close',  # TAQ internal sequence number (trillions range)
    'tr_seqnum_open',   # TAQ internal sequence number (trillions range)
]

# ── 5. Classification-variant duplicates (_tick and _wrds → keep _lr) ───────
# TAQ computes trade direction (buy/sell) using three algorithms.
# For large-cap stocks, all three give nearly identical results.
# Keep Lee-Ready (_lr) as the academic standard, drop _tick and _wrds.

drop_tick_wrds = [
    # Volume by direction
    'buyvol_tick',      'buyvol_wrds',
    'sellvol_tick',     'sellvol_wrds',
    # Number of trades by direction
    'buynumtrades_tick', 'buynumtrades_wrds',
    'sellnumtrades_tick', 'sellnumtrades_wrds',
    # Dollar volume by direction
    'buy_dv_tick',      'buy_dv_wrds',
    'sell_dv_tick',     'sell_dv_wrds',
    # Total trades/volume/dollar (combined buy+sell)
    'total_trade_tick', 'total_trade_wrds',
    'total_dv_tick',    'total_dv_wrds',
    'total_vol_tick',   'total_vol_wrds',
    # Average price by direction
    'avg_buy_price_tick',  'avg_buy_price_wrds',
    'avg_sell_price_tick', 'avg_sell_price_wrds',
    # Volume-weighted average price by direction
    'vwavg_buy_price_tick',  'vwavg_buy_price_wrds',
    'vwavg_sell_price_tick', 'vwavg_sell_price_wrds',
]

# ── 6. Institutional size-threshold duplicates (_inst20k → keep _inst50k) ───
# For top-100 large-caps, $20K and $50K thresholds produce nearly identical
# metrics (r ~ 1.0) because institutional trade sizes far exceed both.
# Keep _inst50k (more selective), drop _inst20k.

drop_inst20k = [
    'buyvol_inst20k',       'sellvol_inst20k',
    'buynumtrades_inst20k', 'sellnumtrades_inst20k',
    'buy_dv_inst20k',       'sell_dv_inst20k',
    'total_trade_inst20k',  'total_dv_inst20k',
    'total_vol_inst20k',
    'avg_buy_price_inst20k', 'avg_sell_price_inst20k',
    'bs_ratio_inst20k_num',  'bs_ratio_inst20k_vol',
]

# ── Combine and execute drops ────────────────────────────────────────────────
all_drops = (drop_non_numeric + drop_exact_dupe + drop_low_var +
             drop_identifiers + drop_tick_wrds + drop_inst20k)

# Only drop columns that actually exist (defensive)
drops_present = [c for c in all_drops if c in df.columns]
drops_missing = [c for c in all_drops if c not in df.columns]

if drops_missing:
    print(f"\n  ⚠ {len(drops_missing)} columns in drop list not found in data:")
    for c in drops_missing:
        print(f"    {c}")

df = df.drop(columns=drops_present)

# ── Report ───────────────────────────────────────────────────────────────────
print(f"\n  Drops by category:")
print(f"    Non-numeric (TAQ metadata):   {len([c for c in drop_non_numeric if c in drops_present]):>3d}")
print(f"    Exact duplicate (dlyclose):   {len([c for c in drop_exact_dupe if c in drops_present]):>3d}")
print(f"    Near-zero variance:           {len([c for c in drop_low_var if c in drops_present]):>3d}")
print(f"    Internal identifiers:         {len([c for c in drop_identifiers if c in drops_present]):>3d}")
print(f"    Classification variants:      {len([c for c in drop_tick_wrds if c in drops_present]):>3d}")
print(f"    Inst size-threshold dupes:    {len([c for c in drop_inst20k if c in drops_present]):>3d}")
print(f"    ──────────────────────────────")
print(f"    Total dropped:                {len(drops_present):>3d}")
print(f"\n  Columns: {n_start} → {df.shape[1]} ({n_start - df.shape[1]} dropped)")

# ── Quick validation: no accidental drops of required columns ────────────────
for c in ['permno', 'date', 'dlyret', 'dlycap']:
    assert c in df.columns, f"FATAL: Required column '{c}' was dropped!"
print(f"  ✓ Required columns (permno, date, dlyret, dlycap) intact")

# ── Check: how many high-correlation pairs remain? ──────────────────────────
remaining_factors = [c for c in df.columns if c not in ['permno', 'date', 'dlyret', 'dlycap']]
remaining_numeric = [c for c in remaining_factors if pd.api.types.is_numeric_dtype(df[c])]
print(f"\n  Remaining factors: {len(remaining_factors)} "
      f"({len(remaining_numeric)} numeric, "
      f"{len(remaining_factors) - len(remaining_numeric)} non-numeric)")

Loaded: 525,957 rows × 240 columns
BLOCK 1: DROP REDUNDANT FACTORS

  Drops by category:
    Non-numeric (TAQ metadata):    13
    Exact duplicate (dlyclose):     1
    Near-zero variance:             2
    Internal identifiers:           2
    Classification variants:       26
    Inst size-threshold dupes:     13
    ──────────────────────────────
    Total dropped:                 57

  Columns: 240 → 183 (57 dropped)
  ✓ Required columns (permno, date, dlyret, dlycap) intact

  Remaining factors: 179 (179 numeric, 0 non-numeric)


In [4]:
# %%
# ── OUTPUT FOR BLOCK 2 PLANNING ──────────────────────────────────────────────
print("=" * 90)
print("SURVIVING COLUMNS FOR BLOCK 2")
print("=" * 90)

remaining = [c for c in df.columns if c not in ['permno', 'date', 'dlyret', 'dlycap']]
remaining_numeric = [c for c in remaining if pd.api.types.is_numeric_dtype(df[c])]

print(f"\n  Total surviving factors: {len(remaining)}")
print(f"\n  Complete list:")
for i, c in enumerate(remaining_numeric, 1):
    vals = df[c].dropna()
    nan_p = df[c].isna().mean() * 100
    print(f"  {i:>4d}. {c:<40s} nan={nan_p:>5.2f}%  "
          f"range=[{vals.min():.4f}, {vals.max():.4f}]")

# Also print remaining non-numeric if any
remaining_non_numeric = [c for c in remaining if c not in remaining_numeric]
if remaining_non_numeric:
    print(f"\n  ⚠ Still have {len(remaining_non_numeric)} non-numeric columns:")
    for c in remaining_non_numeric:
        print(f"    {c}: {df[c].dtype}")

SURVIVING COLUMNS FOR BLOCK 2

  Total surviving factors: 179

  Complete list:
     1. dlyprc                                   nan= 0.00%  range=[0.1300, 5300.3400]
     2. dlyretx                                  nan= 0.00%  range=[-0.9425, 0.9022]
     3. dlyreti                                  nan= 0.00%  range=[0.0000, 0.1028]
     4. dlyvol                                   nan= 0.00%  range=[68373.0000, 3772638437.0000]
     5. dlyopen                                  nan= 0.01%  range=[0.2170, 5300.0000]
     6. dlyhigh                                  nan= 0.00%  range=[0.2300, 5337.2400]
     7. dlylow                                   nan= 0.00%  range=[0.0001, 5260.0000]
     8. dlybid                                   nan= 0.00%  range=[0.1400, 5290.0200]
     9. dlyask                                   nan= 0.00%  range=[0.1450, 5300.7700]
    10. dlyprcvol                                nan= 0.00%  range=[3771398.0000, 98903118748.6000]
    11. shrout                  

In [14]:
# %% [markdown]
# ## Block 2: Complete Factor Inventory
#
# Every surviving factor named, described, sourced, and categorised by scale type.
# Saved as a DataFrame and CSV for reference throughout the pipeline.
#
# Scale types determine how each factor is handled in aggregation:
#   - "ratio"        → already stock-comparable, aggregate directly
#   - "price_level"  → dollar-denominated, must transform to ratio before aggregating
#   - "dollar_volume" → dollar-denominated volume, normalise by market cap before aggregating
#   - "share_volume" → share-denominated volume, normalise by shares outstanding before aggregating
#   - "count"        → trade/quote counts, normalise by total trades before aggregating
#   - "time"         → seconds/timing metric, may need normalisation
#   - "derived"      → computed from other columns (bid_ask_spread)
#   - "exclude"      → do not aggregate (redundant with derived features)

# %%
print("=" * 90)
print("BLOCK 2: COMPLETE FACTOR INVENTORY")
print("=" * 90)

# Build the inventory as a list of dicts, then convert to DataFrame
inventory = []

def add(col, source, category, scale, description):
    inventory.append({
        'column': col,
        'source': source,
        'category': category,
        'scale_type': scale,
        'description': description,
    })

# ═══════════════════════════════════════════════════════════════════════════════
# CRSP FACTORS (12 columns)
# ═══════════════════════════════════════════════════════════════════════════════

add('dlyprc',       'CRSP', 'price',       'price_level',  'Daily closing price')
add('dlyretx',      'CRSP', 'return',      'ratio',        'Daily return excluding dividends (price return)')
add('dlyreti',      'CRSP', 'return',      'ratio',        'Daily return from dividends only (income return)')
add('dlyvol',       'CRSP', 'volume',      'share_volume', 'Daily share volume traded')
add('dlyopen',      'CRSP', 'price',       'price_level',  'Daily opening price')
add('dlyhigh',      'CRSP', 'price',       'price_level',  'Daily high price')
add('dlylow',       'CRSP', 'price',       'price_level',  'Daily low price')
add('dlybid',       'CRSP', 'price',       'price_level',  'Daily closing bid price')
add('dlyask',       'CRSP', 'price',       'price_level',  'Daily closing ask price')
add('dlyprcvol',    'CRSP', 'volume',      'dollar_volume', 'Daily dollar volume (price × volume)')
add('shrout',       'CRSP', 'size',        'exclude',      'Shares outstanding (thousands). Used for normalisation, not as a factor.')
add('bid_ask_spread','CRSP','liquidity',   'ratio',        'Normalised bid-ask spread: |ask - bid| / midpoint. Derived from dlybid/dlyask.')

# ═══════════════════════════════════════════════════════════════════════════════
# TAQ: PRICE LEVELS AT VARIOUS TIMES (will be transformed in Block 3)
# ═══════════════════════════════════════════════════════════════════════════════

# Average trade prices by participant type
add('avg_buy_price_inst50k',  'TAQ', 'price', 'price_level', 'Average buy trade price, institutional trades >$50K')
add('avg_buy_price_lr',       'TAQ', 'price', 'price_level', 'Average buy trade price, Lee-Ready classification')
add('avg_buy_price_retail',   'TAQ', 'price', 'price_level', 'Average buy trade price, retail trades')
add('avg_price_a',            'TAQ', 'price', 'price_level', 'Average trade price, ARCA exchange')
add('avg_price_b',            'TAQ', 'price', 'price_level', 'Average trade price, BATS exchange')
add('avg_price_m',            'TAQ', 'price', 'price_level', 'Average trade price, all exchanges combined')
add('avg_sell_price_inst50k', 'TAQ', 'price', 'price_level', 'Average sell trade price, institutional trades >$50K')
add('avg_sell_price_lr',      'TAQ', 'price', 'price_level', 'Average sell trade price, Lee-Ready classification')
add('avg_sell_price_retail',  'TAQ', 'price', 'price_level', 'Average sell trade price, retail trades')

# NBBO midpoints at specific times
add('mid_1pm',          'TAQ', 'price', 'price_level', 'NBBO midpoint price at 1:00 PM')
add('mid_4pm',          'TAQ', 'price', 'price_level', 'NBBO midpoint price at 4:00 PM')
add('mid_after_open',   'TAQ', 'price', 'price_level', 'NBBO midpoint price shortly after market open')
add('mid_before_close', 'TAQ', 'price', 'price_level', 'NBBO midpoint price shortly before market close')

# National best bid at specific times
add('nbb_1pm',          'TAQ', 'price', 'price_level', 'National best bid at 1:00 PM')
add('nbb_4pm',          'TAQ', 'price', 'price_level', 'National best bid at 4:00 PM')
add('nbb_after_open',   'TAQ', 'price', 'price_level', 'National best bid shortly after market open')
add('nbb_before_close', 'TAQ', 'price', 'price_level', 'National best bid shortly before market close')

# National best offer at specific times
add('nbo_1pm',          'TAQ', 'price', 'price_level', 'National best offer at 1:00 PM')
add('nbo_4pm',          'TAQ', 'price', 'price_level', 'National best offer at 4:00 PM')
add('nbo_after_open',   'TAQ', 'price', 'price_level', 'National best offer shortly after market open')
add('nbo_before_close', 'TAQ', 'price', 'price_level', 'National best offer shortly before market close')

# Pre-trade midpoint at specific times
add('ptime_1pm',   'TAQ', 'price', 'price_level', 'Pre-trade midpoint at 1:00 PM')
add('ptime_4pm',   'TAQ', 'price', 'price_level', 'Pre-trade midpoint at 4:00 PM')
add('ptime_close', 'TAQ', 'price', 'price_level', 'Pre-trade midpoint at close')
add('ptime_open',  'TAQ', 'price', 'price_level', 'Pre-trade midpoint at open')

# TAQ close/open prices
add('cprc',  'TAQ', 'price', 'price_level', 'TAQ closing trade price')
add('oprc',  'TAQ', 'price', 'price_level', 'TAQ opening trade price')

# Volume-weighted prices by venue
add('vw_price_a', 'TAQ', 'price', 'price_level', 'Volume-weighted average trade price, ARCA exchange')
add('vw_price_b', 'TAQ', 'price', 'price_level', 'Volume-weighted average trade price, BATS exchange')
add('vw_price_m', 'TAQ', 'price', 'price_level', 'Volume-weighted average trade price, all exchanges')

# Volume-weighted average prices by direction (Lee-Ready)
add('vwavg_buy_price_lr',  'TAQ', 'price', 'price_level', 'Volume-weighted average buy price, Lee-Ready')
add('vwavg_sell_price_lr', 'TAQ', 'price', 'price_level', 'Volume-weighted average sell price, Lee-Ready')

# Price extremes by venue
add('price_high_a', 'TAQ', 'price', 'price_level', 'Highest trade price, ARCA exchange')
add('price_high_b', 'TAQ', 'price', 'price_level', 'Highest trade price, BATS exchange')
add('price_high_m', 'TAQ', 'price', 'price_level', 'Highest trade price, all exchanges')
add('price_low_a',  'TAQ', 'price', 'price_level', 'Lowest trade price, ARCA exchange')
add('price_low_b',  'TAQ', 'price', 'price_level', 'Lowest trade price, BATS exchange')
add('price_low_m',  'TAQ', 'price', 'price_level', 'Lowest trade price, all exchanges')

# ═══════════════════════════════════════════════════════════════════════════════
# TAQ: SPREADS & PRICE IMPACT (already ratios or small dollar amounts)
# ═══════════════════════════════════════════════════════════════════════════════

# Quoted spread
add('quotedspread_dollar_tw',  'TAQ', 'liquidity', 'ratio', 'Time-weighted quoted spread in dollars (small, ~$0-5)')
add('quotedspread_percent_tw', 'TAQ', 'liquidity', 'ratio', 'Time-weighted quoted spread as percentage of midpoint')

# Effective spread
add('effectivespread_dollar_ave', 'TAQ', 'liquidity', 'ratio', 'Effective spread in dollars, equal-weighted average')
add('effectivespread_dollar_dw',  'TAQ', 'liquidity', 'ratio', 'Effective spread in dollars, dollar-weighted')
add('effectivespread_dollar_sw',  'TAQ', 'liquidity', 'ratio', 'Effective spread in dollars, share-weighted')
add('effectivespread_percent_ave','TAQ', 'liquidity', 'ratio', 'Effective spread as %, equal-weighted average')
add('effectivespread_percent_dw', 'TAQ', 'liquidity', 'ratio', 'Effective spread as %, dollar-weighted')
add('effectivespread_percent_sw', 'TAQ', 'liquidity', 'ratio', 'Effective spread as %, share-weighted')

# Dollar price impact (Lee-Ready)
add('dollarpriceimpact_lr_ave', 'TAQ', 'price_impact', 'ratio', 'Dollar price impact, Lee-Ready, equal-weighted avg')
add('dollarpriceimpact_lr_dw',  'TAQ', 'price_impact', 'ratio', 'Dollar price impact, Lee-Ready, dollar-weighted')
add('dollarpriceimpact_lr_sw',  'TAQ', 'price_impact', 'ratio', 'Dollar price impact, Lee-Ready, share-weighted')

# Dollar realized spread (Lee-Ready)
add('dollarrealizedspread_lr_ave', 'TAQ', 'price_impact', 'ratio', 'Dollar realized spread, Lee-Ready, equal-weighted avg')
add('dollarrealizedspread_lr_dw',  'TAQ', 'price_impact', 'ratio', 'Dollar realized spread, Lee-Ready, dollar-weighted')
add('dollarrealizedspread_lr_sw',  'TAQ', 'price_impact', 'ratio', 'Dollar realized spread, Lee-Ready, share-weighted')

# Percent price impact (Lee-Ready)
add('percentpriceimpact_lr_ave', 'TAQ', 'price_impact', 'ratio', 'Percent price impact, Lee-Ready, equal-weighted avg')
add('percentpriceimpact_lr_dw',  'TAQ', 'price_impact', 'ratio', 'Percent price impact, Lee-Ready, dollar-weighted')
add('percentpriceimpact_lr_sw',  'TAQ', 'price_impact', 'ratio', 'Percent price impact, Lee-Ready, share-weighted')

# Percent realized spread (Lee-Ready)
add('percentrealizedspread_lr_ave', 'TAQ', 'price_impact', 'ratio', 'Percent realized spread, Lee-Ready, equal-weighted avg')
add('percentrealizedspread_lr_dw',  'TAQ', 'price_impact', 'ratio', 'Percent realized spread, Lee-Ready, dollar-weighted')
add('percentrealizedspread_lr_sw',  'TAQ', 'price_impact', 'ratio', 'Percent realized spread, Lee-Ready, share-weighted')

# ═══════════════════════════════════════════════════════════════════════════════
# TAQ: ORDER BOOK DEPTH (dollar or share denominated)
# ═══════════════════════════════════════════════════════════════════════════════

add('bestbiddepth_dollar_tw',  'TAQ', 'depth', 'dollar_volume', 'Time-weighted best bid depth in dollars')
add('bestbiddepth_share_tw',   'TAQ', 'depth', 'share_volume',  'Time-weighted best bid depth in shares')
add('bestofrdepth_dollar_tw',  'TAQ', 'depth', 'dollar_volume', 'Time-weighted best offer depth in dollars')
add('bestofrdepth_share_tw',   'TAQ', 'depth', 'share_volume',  'Time-weighted best offer depth in shares')

# NBBO quantities at specific times
add('nbbqty_1pm',          'TAQ', 'depth', 'share_volume', 'National best bid quantity at 1:00 PM')
add('nbbqty_4pm',          'TAQ', 'depth', 'share_volume', 'National best bid quantity at 4:00 PM')
add('nbbqty_after_open',   'TAQ', 'depth', 'share_volume', 'National best bid quantity after open')
add('nbbqty_before_close', 'TAQ', 'depth', 'share_volume', 'National best bid quantity before close')
add('nboqty_1pm',          'TAQ', 'depth', 'share_volume', 'National best offer quantity at 1:00 PM')
add('nboqty_4pm',          'TAQ', 'depth', 'share_volume', 'National best offer quantity at 4:00 PM')
add('nboqty_after_open',   'TAQ', 'depth', 'share_volume', 'National best offer quantity after open')
add('nboqty_before_close', 'TAQ', 'depth', 'share_volume', 'National best offer quantity before close')

# ═══════════════════════════════════════════════════════════════════════════════
# TAQ: BUY-SELL RATIOS (already bounded 0–1, stock-comparable)
# ═══════════════════════════════════════════════════════════════════════════════

add('bs_ratio_inst50k_num',  'TAQ', 'order_flow', 'ratio', 'Buy-sell ratio by trade count, institutional >$50K. 0.5 = balanced.')
add('bs_ratio_inst50k_vol',  'TAQ', 'order_flow', 'ratio', 'Buy-sell ratio by volume, institutional >$50K. 0.5 = balanced.')
add('bs_ratio_num',          'TAQ', 'order_flow', 'ratio', 'Buy-sell ratio by trade count, all trades. 0.5 = balanced.')
add('bs_ratio_retail_num',   'TAQ', 'order_flow', 'ratio', 'Buy-sell ratio by trade count, retail trades. 0.5 = balanced.')
add('bs_ratio_retail_vol',   'TAQ', 'order_flow', 'ratio', 'Buy-sell ratio by volume, retail trades. 0.5 = balanced.')
add('bs_ratio_vol',          'TAQ', 'order_flow', 'ratio', 'Buy-sell ratio by volume, all trades. 0.5 = balanced.')

# ═══════════════════════════════════════════════════════════════════════════════
# TAQ: DOLLAR VOLUME BY DIRECTION & PARTICIPANT (needs market cap normalisation)
# ═══════════════════════════════════════════════════════════════════════════════

add('buy_dv_inst50k',  'TAQ', 'order_flow', 'dollar_volume', 'Buy-side dollar volume, institutional trades >$50K')
add('buy_dv_lr',       'TAQ', 'order_flow', 'dollar_volume', 'Buy-side dollar volume, Lee-Ready classification')
add('buy_dv_retail',   'TAQ', 'order_flow', 'dollar_volume', 'Buy-side dollar volume, retail trades')
add('sell_dv_inst50k', 'TAQ', 'order_flow', 'dollar_volume', 'Sell-side dollar volume, institutional trades >$50K')
add('sell_dv_lr',      'TAQ', 'order_flow', 'dollar_volume', 'Sell-side dollar volume, Lee-Ready classification')
add('sell_dv_retail',  'TAQ', 'order_flow', 'dollar_volume', 'Sell-side dollar volume, retail trades')
add('total_dv_inst50k','TAQ', 'order_flow', 'dollar_volume', 'Total dollar volume, institutional trades >$50K')
add('total_dv_lr',     'TAQ', 'order_flow', 'dollar_volume', 'Total dollar volume, Lee-Ready classification')
add('total_dv_retail', 'TAQ', 'order_flow', 'dollar_volume', 'Total dollar volume, retail trades')

# Dollar volume by venue
add('total_dollar_a', 'TAQ', 'venue', 'dollar_volume', 'Total dollar volume, ARCA exchange')
add('total_dollar_b', 'TAQ', 'venue', 'dollar_volume', 'Total dollar volume, BATS exchange')
add('total_dollar_m', 'TAQ', 'venue', 'dollar_volume', 'Total dollar volume, all exchanges')

# ISO dollar volume
add('iso_dollar', 'TAQ', 'order_flow', 'dollar_volume', 'Intermarket sweep order dollar volume')

# ═══════════════════════════════════════════════════════════════════════════════
# TAQ: SHARE VOLUME BY DIRECTION & PARTICIPANT (needs shares outstanding normalisation)
# ═══════════════════════════════════════════════════════════════════════════════

add('buyvol_inst50k',  'TAQ', 'order_flow', 'share_volume', 'Buy share volume, institutional trades >$50K')
add('buyvol_lr',       'TAQ', 'order_flow', 'share_volume', 'Buy share volume, Lee-Ready classification')
add('buyvol_retail',   'TAQ', 'order_flow', 'share_volume', 'Buy share volume, retail trades')
add('sellvol_inst50k', 'TAQ', 'order_flow', 'share_volume', 'Sell share volume, institutional trades >$50K')
add('sellvol_lr',      'TAQ', 'order_flow', 'share_volume', 'Sell share volume, Lee-Ready classification')
add('sellvol_retail',  'TAQ', 'order_flow', 'share_volume', 'Sell share volume, retail trades')
add('total_vol',       'TAQ', 'volume',     'share_volume', 'Total share volume, all trades')
add('total_vol_a',     'TAQ', 'venue',      'share_volume', 'Total share volume, ARCA exchange')
add('total_vol_b',     'TAQ', 'venue',      'share_volume', 'Total share volume, BATS exchange')
add('total_vol_inst50k','TAQ','order_flow',  'share_volume', 'Total share volume, institutional trades >$50K')
add('total_vol_m',     'TAQ', 'venue',      'share_volume', 'Total share volume, all exchanges (TAQ)')
add('total_vol_retail','TAQ', 'order_flow',  'share_volume', 'Total share volume, retail trades')
add('iso_vol',         'TAQ', 'order_flow',  'share_volume', 'Intermarket sweep order share volume')

# ═══════════════════════════════════════════════════════════════════════════════
# TAQ: TRADE COUNTS BY DIRECTION & PARTICIPANT (needs normalisation)
# ═══════════════════════════════════════════════════════════════════════════════

add('buynumtrades_inst50k',  'TAQ', 'order_flow', 'count', 'Number of buy trades, institutional >$50K')
add('buynumtrades_lr',       'TAQ', 'order_flow', 'count', 'Number of buy trades, Lee-Ready')
add('buynumtrades_retail',   'TAQ', 'order_flow', 'count', 'Number of buy trades, retail')
add('sellnumtrades_inst50k', 'TAQ', 'order_flow', 'count', 'Number of sell trades, institutional >$50K')
add('sellnumtrades_lr',      'TAQ', 'order_flow', 'count', 'Number of sell trades, Lee-Ready')
add('sellnumtrades_retail',  'TAQ', 'order_flow', 'count', 'Number of sell trades, retail')
add('total_trade',           'TAQ', 'volume',     'count', 'Total number of trades, all types')
add('total_trade_inst50k',   'TAQ', 'order_flow', 'count', 'Total number of trades, institutional >$50K')
add('total_trade_retail',    'TAQ', 'order_flow', 'count', 'Total number of trades, retail')
add('total_n_trades_a',      'TAQ', 'venue',      'count', 'Total number of trades, ARCA exchange')
add('total_n_trades_b',      'TAQ', 'venue',      'count', 'Total number of trades, BATS exchange')
add('total_n_trades_m',      'TAQ', 'venue',      'count', 'Total number of trades, all exchanges')
add('n_iso_trade',           'TAQ', 'order_flow', 'count', 'Number of intermarket sweep order trades')
add('n_outside_nbbo_trade',  'TAQ', 'order_flow', 'count', 'Number of trades executed outside NBBO')

# ═══════════════════════════════════════════════════════════════════════════════
# TAQ: TRADE SIZE & TIMING
# ═══════════════════════════════════════════════════════════════════════════════

add('csize',      'TAQ', 'trade_size', 'share_volume', 'Closing trade size in shares')
add('osize',      'TAQ', 'trade_size', 'share_volume', 'Opening trade size in shares')
add('size_1pm',   'TAQ', 'trade_size', 'share_volume', 'Trade size at 1:00 PM in shares')
add('size_4pm',   'TAQ', 'trade_size', 'share_volume', 'Trade size at 4:00 PM in shares')
add('stime_close','TAQ', 'timing',     'time',         'Seconds from open to last trade of the day')
add('stime_open', 'TAQ', 'timing',     'time',         'Seconds from midnight to first trade of the day')

# ═══════════════════════════════════════════════════════════════════════════════
# TAQ: MICROSTRUCTURE QUALITY METRICS (already ratios/bounded)
# ═══════════════════════════════════════════════════════════════════════════════

add('hindex',     'TAQ', 'concentration', 'ratio', 'Herfindahl index of trade size concentration. 0 = dispersed, 1 = concentrated.')
add('ivol_q',     'TAQ', 'volatility',    'ratio', 'Intraday quote-midpoint volatility (variance of quote midpoint returns)')
add('ivol_t',     'TAQ', 'volatility',    'ratio', 'Intraday trade-price volatility (variance of trade price returns)')
add('n30_pos',    'TAQ', 'momentum',      'ratio', 'Number of 30-second intervals with positive midpoint returns')
add('n5_pos',     'TAQ', 'momentum',      'ratio', 'Number of 5-minute intervals with positive midpoint returns')
add('n_obs',      'TAQ', 'coverage',      'ratio', 'Number of observation intervals with valid quotes')
add('ret_mkt_m',  'TAQ', 'return',        'ratio', 'Intraday return computed from midpoint prices (open to close)')

# Trade sign metrics (Kyle lambda proxies)
add('tsignsqrtdvol1', 'TAQ', 'price_impact', 'ratio', 'Trade sign × sqrt(dollar volume), metric 1 (Kyle lambda proxy)')
add('tsignsqrtdvol2', 'TAQ', 'price_impact', 'ratio', 'Trade sign × sqrt(dollar volume), metric 2 (Kyle lambda proxy)')

# Variance ratios (market efficiency)
add('var_ratio1', 'TAQ', 'efficiency', 'ratio', 'Variance ratio at horizon 1 (1 = efficient, <1 = mean-reverting, >1 = trending)')
add('var_ratio2', 'TAQ', 'efficiency', 'ratio', 'Variance ratio at horizon 2')
add('var_ratio3', 'TAQ', 'efficiency', 'ratio', 'Variance ratio at horizon 3')
add('var_ratio4', 'TAQ', 'efficiency', 'ratio', 'Variance ratio at horizon 4')
add('var_ratio5', 'TAQ', 'efficiency', 'ratio', 'Variance ratio at horizon 5')

# ═══════════════════════════════════════════════════════════════════════════════
# OPTIONMETRICS: IMPLIED VOLATILITY & SURFACE
# ═══════════════════════════════════════════════════════════════════════════════

add('iv_catm',            'OM', 'volatility',     'ratio', 'Implied volatility, call ATM, ~30-day tenor (annualised decimal)')
add('iv_PATM',            'OM', 'volatility',     'ratio', 'Implied volatility, put ATM, ~30-day tenor')
add('iv_POTM',            'OM', 'volatility',     'ratio', 'Implied volatility, put OTM (out-of-the-money)')
add('iv_91d_atm',         'OM', 'volatility',     'ratio', 'Implied volatility, ATM, 91-day tenor')
add('iv_30d_call25',      'OM', 'volatility',     'ratio', 'Implied volatility, 30-day, 25-delta OTM call')
add('iv_30d_put25',       'OM', 'volatility',     'ratio', 'Implied volatility, 30-day, 25-delta OTM put')
add('vol_term_structure',  'OM', 'vol_surface',   'ratio', 'IV term structure: 91d ATM minus 30d ATM. Positive = contango.')
add('vol_smile',           'OM', 'vol_surface',   'ratio', 'Volatility smile: put25 + call25 - 2×ATM. Positive = convex smile.')
add('Skew_OTM',           'OM', 'vol_surface',    'ratio', 'OTM skew: IV_POTM minus IV_CATM. Measures put premium over call.')

# ═══════════════════════════════════════════════════════════════════════════════
# OPTIONMETRICS: REALISED VOL & VOLATILITY RISK PREMIUM
# ═══════════════════════════════════════════════════════════════════════════════

add('hvol',     'OM', 'volatility', 'ratio', 'Historical (realised) volatility, 30-day, from OptionMetrics')
add('rv_30d',   'OM', 'volatility', 'ratio', 'Realised volatility, 30-day, computed from CRSP daily returns')
add('vrp_rv',   'OM', 'volatility', 'ratio', 'Volatility risk premium: IV_CATM minus RV_30d. Positive = IV > realised.')
add('vrp_hvol', 'OM', 'volatility', 'ratio', 'Volatility risk premium: IV_CATM minus hvol. Positive = IV > historical.')

# ═══════════════════════════════════════════════════════════════════════════════
# OPTIONMETRICS: PUT-CALL METRICS
# ═══════════════════════════════════════════════════════════════════════════════

add('Parity_VSpread', 'OM', 'options',    'ratio', 'Put-call parity violation spread. Measures options mispricing.')
add('nopt_Parity',    'OM', 'options',    'count', 'Number of option pairs used in parity calculation')
add('PC_Ratio',       'OM', 'sentiment',  'ratio', 'Put-call volume ratio. >1 = more puts (bearish), <1 = more calls (bullish).')

# ═══════════════════════════════════════════════════════════════════════════════
# OPTIONMETRICS: GREEKS & POSITIONING (already normalised by market cap)
# ═══════════════════════════════════════════════════════════════════════════════

add('oi_wt_delta',             'OM', 'positioning', 'ratio', 'Open-interest-weighted average delta. Net directional tilt of options market.')
add('oi_wt_gamma',             'OM', 'positioning', 'ratio', 'Open-interest-weighted average gamma. Convexity exposure of options market.')
add('oi_wt_vega',              'OM', 'positioning', 'ratio', 'Open-interest-weighted average vega. Sensitivity to implied vol changes.')
add('oi_wt_theta',             'OM', 'positioning', 'ratio', 'Open-interest-weighted average theta. Time decay exposure (always negative).')
add('gex_norm',                'OM', 'positioning', 'ratio', 'Gamma exposure (calls minus puts) normalised by market cap. Measures dealer hedging pressure.')
add('dex_norm',                'OM', 'positioning', 'ratio', 'Delta exposure normalised by market cap. Net directional positioning.')
add('delta_dollar_volume_norm','OM', 'positioning', 'ratio', 'Delta-weighted dollar volume normalised by market cap. Flow toxicity measure.')
add('total_oi_norm',           'OM', 'positioning', 'ratio', 'Total open interest normalised by market cap. Options market depth.')
add('total_volume_norm',       'OM', 'positioning', 'ratio', 'Total options volume normalised by market cap. Options activity level.')

# ═══════════════════════════════════════════════════════════════════════════════
# OPTIONMETRICS: MONEYNESS DISTRIBUTION OF OPEN INTEREST
# ═══════════════════════════════════════════════════════════════════════════════

add('sumOI_c_money1_pct', 'OM', 'positioning', 'ratio', 'Call OI in moneyness bucket 1 (near ATM) as % of total OI')
add('sumOI_c_money2_pct', 'OM', 'positioning', 'ratio', 'Call OI in moneyness bucket 2 (moderate OTM) as % of total OI')
add('sumOI_c_money3_pct', 'OM', 'positioning', 'ratio', 'Call OI in moneyness bucket 3 (deep OTM) as % of total OI')
add('sumOI_p_money1_pct', 'OM', 'positioning', 'ratio', 'Put OI in moneyness bucket 1 (near ATM) as % of total OI')
add('sumOI_p_money2_pct', 'OM', 'positioning', 'ratio', 'Put OI in moneyness bucket 2 (moderate OTM) as % of total OI')
add('sumOI_p_money3_pct', 'OM', 'positioning', 'ratio', 'Put OI in moneyness bucket 3 (deep OTM) as % of total OI')

# ═══════════════════════════════════════════════════════════════════════════════
# BUILD AND VALIDATE THE INVENTORY DATAFRAME
# ═══════════════════════════════════════════════════════════════════════════════

# %%
inv = pd.DataFrame(inventory)

# Verify every surviving factor is catalogued
remaining_factors = [c for c in df.columns if c not in ['permno', 'date', 'dlyret', 'dlycap']]
catalogued = set(inv['column'])
in_data_not_catalogued = [c for c in remaining_factors if c not in catalogued]
in_catalogue_not_data = [c for c in catalogued if c not in remaining_factors]

print(f"\n  Factors in data: {len(remaining_factors)}")
print(f"  Factors catalogued: {len(inv)}")

if in_data_not_catalogued:
    print(f"\n  ✗ MISSING from catalogue ({len(in_data_not_catalogued)}):")
    for c in in_data_not_catalogued:
        print(f"    {c}")
else:
    print(f"  ✓ Every factor in data is catalogued")

if in_catalogue_not_data:
    print(f"\n  ✗ In catalogue but NOT in data ({len(in_catalogue_not_data)}):")
    for c in in_catalogue_not_data:
        print(f"    {c}")
else:
    print(f"  ✓ Every catalogued factor exists in data")

# Summary by source and scale type
print(f"\n  By source:")
print(inv['source'].value_counts().to_string())

print(f"\n  By scale type:")
print(inv['scale_type'].value_counts().to_string())

print(f"\n  By category:")
print(inv['category'].value_counts().to_string())

# ═══════════════════════════════════════════════════════════════════════════════
# SAVE INVENTORY
# ═══════════════════════════════════════════════════════════════════════════════

# %%
OUT_DIR = Path('../../../Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering')
OUT_DIR.mkdir(parents=True, exist_ok=True)
inv_path = OUT_DIR/'stock_daily_descriptions_pre.csv'
inv.to_csv(str(inv_path), index=False)
print(f"\n  ✓ Saved: {inv_path}")
print(f"    {len(inv)} factors catalogued")

# Also print the full table
print(f"\n  Complete inventory:")
print(f"  {'#':<4s} {'Column':<35s} {'Source':<6s} {'Scale':<14s} {'Category':<14s} Description")
print("  " + "-" * 120)
for i, row in inv.iterrows():
    print(f"  {i+1:<4d} {row['column']:<35s} {row['source']:<6s} "
          f"{row['scale_type']:<14s} {row['category']:<14s} {row['description']}")

BLOCK 2: COMPLETE FACTOR INVENTORY

  Factors in data: 179
  Factors catalogued: 179
  ✓ Every factor in data is catalogued
  ✓ Every catalogued factor exists in data

  By source:
source
TAQ     136
OM       31
CRSP     12

  By scale type:
scale_type
ratio            73
price_level      44
share_volume     28
dollar_volume    16
count            15
time              2
exclude           1

  By category:
category
price            44
order_flow       35
positioning      15
price_impact     14
depth            12
volatility       12
venue             9
liquidity         9
efficiency        5
volume            4
trade_size        4
return            3
vol_surface       3
timing            2
options           2
momentum          2
sentiment         1
size              1
coverage          1
concentration     1

  ✓ Saved: ..\..\..\Data\Data_Collection\Final\Stage_1_5_Validation_and_Feature_Engineering\stock_daily_descriptions_pre.csv
    179 factors catalogued

  Complete inventory:
  #   

In [15]:
# %% [markdown]
# ## Block 3: Feature Engineering
#
# Four sections, executed in order:
#   A. Derive new ratio features from price-level column PAIRS
#   B. Normalise dollar volumes by market cap, share volumes by shares
#      outstanding, and trade counts by total trades
#   C. Rolling & dynamic features — momentum, volatility dynamics,
#      volume surprises, liquidity shocks, IV dynamics, flow persistence
#   D. Drop original raw columns replaced by derived/normalised versions
#
# After this block, every factor is stock-comparable and ready for
# cross-sectional aggregation in Step 2.

# %%
print("=" * 90)
print("BLOCK 3: FEATURE ENGINEERING")
print("=" * 90)

n_before = df.shape[1]
new_features = []

# Ensure sorted by permno+date for rolling computations
df = df.sort_values(['permno', 'date']).reset_index(drop=True)

# Helper: safe division (returns NaN when denominator is 0 or NaN)
def safe_div(num, denom):
    return num / denom.replace(0, np.nan)

# Helper: per-stock rolling with min_periods safety
def grp_roll(col, window, func='mean', min_periods=None):
    """Compute rolling stat per permno. Returns a Series aligned to df."""
    if min_periods is None:
        min_periods = max(window // 2, 1)
    r = df.groupby('permno')[col].transform(
        lambda x: getattr(x.rolling(window, min_periods=min_periods), func)()
    )
    return r

# Helper: per-stock diff
def grp_diff(col, periods=1):
    return df.groupby('permno')[col].transform(lambda x: x.diff(periods))

# Helper: per-stock shift
def grp_shift(col, periods=1):
    return df.groupby('permno')[col].transform(lambda x: x.shift(periods))

# ═══════════════════════════════════════════════════════════════════════════════
# A. DERIVED RATIO FEATURES FROM PRICE-LEVEL PAIRS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n--- A. Derived ratio features from price-level pairs ---\n")

a_start = len(new_features)

# ── A1. CRSP intraday features ──────────────────────────────────────────────

df['intraday_range'] = safe_div(df['dlyhigh'] - df['dlylow'], df['dlyprc'])
new_features.append('intraday_range')

df['open_to_close_ret'] = safe_div(df['dlyprc'] - df['dlyopen'], df['dlyopen'])
new_features.append('open_to_close_ret')

df['turnover'] = safe_div(df['dlyvol'], df['shrout'])
new_features.append('turnover')

df['dvol_to_cap'] = safe_div(df['dlyprcvol'], df['dlycap'])
new_features.append('dvol_to_cap')

# ── A2. TAQ buy-sell price premiums ─────────────────────────────────────────

df['buysell_premium_lr'] = safe_div(
    df['avg_buy_price_lr'] - df['avg_sell_price_lr'], df['dlyprc']
)
new_features.append('buysell_premium_lr')

df['buysell_premium_inst50k'] = safe_div(
    df['avg_buy_price_inst50k'] - df['avg_sell_price_inst50k'], df['dlyprc']
)
new_features.append('buysell_premium_inst50k')

df['buysell_premium_retail'] = safe_div(
    df['avg_buy_price_retail'] - df['avg_sell_price_retail'], df['dlyprc']
)
new_features.append('buysell_premium_retail')

# ── A3. TAQ venue-level intraday ranges ─────────────────────────────────────

df['venue_range_m'] = safe_div(df['price_high_m'] - df['price_low_m'], df['dlyprc'])
new_features.append('venue_range_m')

df['venue_range_a'] = safe_div(df['price_high_a'] - df['price_low_a'], df['dlyprc'])
new_features.append('venue_range_a')

df['venue_range_b'] = safe_div(df['price_high_b'] - df['price_low_b'], df['dlyprc'])
new_features.append('venue_range_b')

# ── A4. TAQ intraday price drift ────────────────────────────────────────────

df['intraday_drift'] = safe_div(df['mid_4pm'] - df['mid_after_open'], df['mid_after_open'])
new_features.append('intraday_drift')

df['midday_drift'] = safe_div(df['mid_4pm'] - df['mid_1pm'], df['mid_1pm'])
new_features.append('midday_drift')

df['morning_drift'] = safe_div(df['mid_1pm'] - df['mid_after_open'], df['mid_after_open'])
new_features.append('morning_drift')

# ── A5. NBBO spread at different times of day ───────────────────────────────

df['nbbo_spread_1pm'] = safe_div(df['nbo_1pm'] - df['nbb_1pm'], df['mid_1pm'])
new_features.append('nbbo_spread_1pm')

df['nbbo_spread_4pm'] = safe_div(df['nbo_4pm'] - df['nbb_4pm'], df['mid_4pm'])
new_features.append('nbbo_spread_4pm')

df['nbbo_spread_open'] = safe_div(
    df['nbo_after_open'] - df['nbb_after_open'], df['mid_after_open']
)
new_features.append('nbbo_spread_open')

df['nbbo_spread_close'] = safe_div(
    df['nbo_before_close'] - df['nbb_before_close'], df['mid_before_close']
)
new_features.append('nbbo_spread_close')

# ── A6. Close/open price positioning vs midpoint ────────────────────────────

df['close_vs_mid'] = safe_div(df['cprc'] - df['mid_4pm'], df['mid_4pm'])
new_features.append('close_vs_mid')

df['open_vs_mid'] = safe_div(df['oprc'] - df['mid_after_open'], df['mid_after_open'])
new_features.append('open_vs_mid')

# ── A7. VWAP-to-close deviation ─────────────────────────────────────────────

df['vwap_deviation_m'] = safe_div(df['vw_price_m'] - df['dlyprc'], df['dlyprc'])
new_features.append('vwap_deviation_m')

# ── A8. Depth imbalance (from raw share depth, ratio cancels units) ─────────

df['depth_imbalance'] = safe_div(
    df['bestbiddepth_share_tw'] - df['bestofrdepth_share_tw'],
    df['bestbiddepth_share_tw'] + df['bestofrdepth_share_tw']
)
new_features.append('depth_imbalance')

print(f"  Section A: {len(new_features) - a_start} price-derived features created")

# ═══════════════════════════════════════════════════════════════════════════════
# B. NORMALISE VOLUME, DOLLAR VOLUME, AND COUNT COLUMNS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n--- B. Normalise volume, dollar volume, and count columns ---\n")

b_start = len(new_features)

# ── B1. Dollar volumes → divide by dlycap ───────────────────────────────────
dollar_vol_cols = [
    'buy_dv_inst50k', 'buy_dv_lr', 'buy_dv_retail',
    'sell_dv_inst50k', 'sell_dv_lr', 'sell_dv_retail',
    'total_dv_inst50k', 'total_dv_lr', 'total_dv_retail',
    'total_dollar_a', 'total_dollar_b', 'total_dollar_m',
    'iso_dollar',
    'bestbiddepth_dollar_tw', 'bestofrdepth_dollar_tw',
]

dv_count = 0
for col in dollar_vol_cols:
    if col in df.columns:
        new_name = f"{col}_to_cap"
        df[new_name] = safe_div(df[col], df['dlycap'])
        new_features.append(new_name)
        dv_count += 1

print(f"  Dollar volume → /dlycap: {dv_count} normalised")

# ── B2. Share volumes → divide by shrout ────────────────────────────────────
share_vol_cols = [
    'buyvol_inst50k', 'buyvol_lr', 'buyvol_retail',
    'sellvol_inst50k', 'sellvol_lr', 'sellvol_retail',
    'total_vol', 'total_vol_a', 'total_vol_b',
    'total_vol_inst50k', 'total_vol_m', 'total_vol_retail',
    'iso_vol',
    'bestbiddepth_share_tw', 'bestofrdepth_share_tw',
    'nbbqty_1pm', 'nbbqty_4pm', 'nbbqty_after_open', 'nbbqty_before_close',
    'nboqty_1pm', 'nboqty_4pm', 'nboqty_after_open', 'nboqty_before_close',
    'csize', 'osize', 'size_1pm', 'size_4pm',
]

sv_count = 0
for col in share_vol_cols:
    if col in df.columns:
        new_name = f"{col}_to_shrout"
        df[new_name] = safe_div(df[col], df['shrout'])
        new_features.append(new_name)
        sv_count += 1

print(f"  Share volume → /shrout: {sv_count} normalised")

# ── B3. Trade counts → divide by total_trade ────────────────────────────────
count_cols = [
    'buynumtrades_inst50k', 'buynumtrades_lr', 'buynumtrades_retail',
    'sellnumtrades_inst50k', 'sellnumtrades_lr', 'sellnumtrades_retail',
    'total_trade_inst50k', 'total_trade_retail',
    'total_n_trades_a', 'total_n_trades_b', 'total_n_trades_m',
    'n_iso_trade', 'n_outside_nbbo_trade',
]

ct_count = 0
for col in count_cols:
    if col in df.columns:
        new_name = f"{col}_pct"
        df[new_name] = safe_div(df[col], df['total_trade'])
        new_features.append(new_name)
        ct_count += 1

print(f"  Trade counts → /total_trade: {ct_count} normalised")
print(f"  Section B total: {len(new_features) - b_start} normalised features")

# ═══════════════════════════════════════════════════════════════════════════════
# C. ROLLING & DYNAMIC FEATURES (PER-STOCK)
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n--- C. Rolling & dynamic features (per-stock) ---\n")
print("  Computing rolling features (this may take a few minutes)...\n")

c_start = len(new_features)

# ── C1. RETURN MOMENTUM & REVERSAL ──────────────────────────────────────────

# Cumulative returns over various horizons
df['ret_cum_5d'] = grp_roll('dlyretx', 5, 'sum')
new_features.append('ret_cum_5d')

df['ret_cum_20d'] = grp_roll('dlyretx', 20, 'sum')
new_features.append('ret_cum_20d')

# Extreme returns in recent window (attention/tail risk signals)
df['ret_max_5d'] = grp_roll('dlyretx', 5, 'max')
new_features.append('ret_max_5d')

df['ret_min_5d'] = grp_roll('dlyretx', 5, 'min')
new_features.append('ret_min_5d')

print(f"  C1 Return momentum: 4 features")

# ── C2. RETURN VOLATILITY DYNAMICS ──────────────────────────────────────────

df['ret_vol_5d'] = grp_roll('dlyretx', 5, 'std')
new_features.append('ret_vol_5d')

df['ret_vol_20d'] = grp_roll('dlyretx', 20, 'std')
new_features.append('ret_vol_20d')

# Vol regime detection: short-term vol / long-term vol
# >1 means vol is spiking relative to recent norm
df['ret_vol_ratio'] = safe_div(df['ret_vol_5d'], df['ret_vol_20d'])
new_features.append('ret_vol_ratio')

print(f"  C2 Return volatility: 3 features")

# ── C3. VOLUME DYNAMICS (from already-derived turnover and dvol_to_cap) ─────

# Smoothed levels
df['turnover_5d_mean'] = grp_roll('turnover', 5)
new_features.append('turnover_5d_mean')

df['turnover_20d_mean'] = grp_roll('turnover', 20)
new_features.append('turnover_20d_mean')

# Volume surprise: today vs recent average
# >1 means unusually high volume today
df['turnover_rel_5d'] = safe_div(df['turnover'], df['turnover_5d_mean'])
new_features.append('turnover_rel_5d')

df['turnover_rel_20d'] = safe_div(df['turnover'], df['turnover_20d_mean'])
new_features.append('turnover_rel_20d')

# Dollar volume intensity surprise
dvol_5d = grp_roll('dvol_to_cap', 5)
df['dvol_to_cap_rel_5d'] = safe_div(df['dvol_to_cap'], dvol_5d)
new_features.append('dvol_to_cap_rel_5d')

print(f"  C3 Volume dynamics: 5 features")

# ── C4. LIQUIDITY DYNAMICS (from bid_ask_spread) ────────────────────────────

df['spread_5d_mean'] = grp_roll('bid_ask_spread', 5)
new_features.append('spread_5d_mean')

df['spread_20d_mean'] = grp_roll('bid_ask_spread', 20)
new_features.append('spread_20d_mean')

# Liquidity shock: today's spread / recent average
# >1 means liquidity is deteriorating
df['spread_rel_5d'] = safe_div(df['bid_ask_spread'], df['spread_5d_mean'])
new_features.append('spread_rel_5d')

df['spread_rel_20d'] = safe_div(df['bid_ask_spread'], df['spread_20d_mean'])
new_features.append('spread_rel_20d')

# Effective spread surprise (TAQ, more precise than CRSP bid-ask)
if 'effectivespread_percent_ave' in df.columns:
    eff_5d = grp_roll('effectivespread_percent_ave', 5)
    df['effspread_pct_rel_5d'] = safe_div(df['effectivespread_percent_ave'], eff_5d)
    new_features.append('effspread_pct_rel_5d')

print(f"  C4 Liquidity dynamics: 5 features")

# ── C5. INTRADAY PATTERN DYNAMICS ───────────────────────────────────────────

# Range surprise: today's range / recent average
range_5d = grp_roll('intraday_range', 5)
df['intraday_range_rel_5d'] = safe_div(df['intraday_range'], range_5d)
new_features.append('intraday_range_rel_5d')

# Smoothed range level
df['intraday_range_5d_mean'] = range_5d
new_features.append('intraday_range_5d_mean')

# Sustained intraday direction
df['open_to_close_ret_5d_mean'] = grp_roll('open_to_close_ret', 5)
new_features.append('open_to_close_ret_5d_mean')

print(f"  C5 Intraday dynamics: 3 features")

# ── C6. IMPLIED VOLATILITY DYNAMICS ─────────────────────────────────────────

if 'iv_catm' in df.columns:
    # IV momentum (1-day and 5-day changes)
    df['iv_catm_chg_1d'] = grp_diff('iv_catm', 1)
    new_features.append('iv_catm_chg_1d')

    df['iv_catm_chg_5d'] = grp_diff('iv_catm', 5)
    new_features.append('iv_catm_chg_5d')

    # IV relative to 20-day mean (IV regime)
    iv_20d = grp_roll('iv_catm', 20)
    df['iv_catm_rel_20d'] = safe_div(df['iv_catm'], iv_20d)
    new_features.append('iv_catm_rel_20d')

    # VRP dynamics
    if 'vrp_rv' in df.columns:
        df['vrp_rv_chg_5d'] = grp_diff('vrp_rv', 5)
        new_features.append('vrp_rv_chg_5d')

    # Skew dynamics (tail fear shift)
    if 'Skew_OTM' in df.columns:
        df['skew_chg_5d'] = grp_diff('Skew_OTM', 5)
        new_features.append('skew_chg_5d')

    # Vol term structure dynamics
    if 'vol_term_structure' in df.columns:
        df['vol_term_chg_5d'] = grp_diff('vol_term_structure', 5)
        new_features.append('vol_term_chg_5d')

    iv_count = sum(1 for f in new_features[c_start:] if 'iv_' in f or 'vrp_' in f or 'skew_' in f or 'vol_term_' in f)
    print(f"  C6 IV dynamics: {iv_count} features")
else:
    print(f"  C6 IV dynamics: skipped (iv_catm not present)")

# ── C7. ORDER FLOW DYNAMICS ─────────────────────────────────────────────────

# Buy-sell ratio persistence and shifts
if 'bs_ratio_vol' in df.columns:
    df['bs_ratio_vol_5d_mean'] = grp_roll('bs_ratio_vol', 5)
    new_features.append('bs_ratio_vol_5d_mean')

    df['bs_ratio_vol_chg_1d'] = grp_diff('bs_ratio_vol', 1)
    new_features.append('bs_ratio_vol_chg_1d')

if 'bs_ratio_inst50k_vol' in df.columns:
    df['bs_ratio_inst_5d_mean'] = grp_roll('bs_ratio_inst50k_vol', 5)
    new_features.append('bs_ratio_inst_5d_mean')

    df['bs_ratio_inst_chg_5d'] = grp_diff('bs_ratio_inst50k_vol', 5)
    new_features.append('bs_ratio_inst_chg_5d')

# Retail participation dynamics (from normalised retail volume)
if 'total_vol_retail_to_shrout' in df.columns:
    retail_5d = grp_roll('total_vol_retail_to_shrout', 5)
    df['retail_vol_5d_mean'] = retail_5d
    new_features.append('retail_vol_5d_mean')

print(f"  C7 Order flow dynamics: {sum(1 for f in ['bs_ratio_vol_5d_mean','bs_ratio_vol_chg_1d','bs_ratio_inst_5d_mean','bs_ratio_inst_chg_5d','retail_vol_5d_mean'] if f in df.columns)} features")

# ── C8. OPTIONS POSITIONING DYNAMICS ────────────────────────────────────────

if 'PC_Ratio' in df.columns:
    # Put-call ratio dynamics (sentiment shift)
    df['pc_ratio_chg_5d'] = grp_diff('PC_Ratio', 5)
    new_features.append('pc_ratio_chg_5d')

    pc_20d = grp_roll('PC_Ratio', 20)
    df['pc_ratio_rel_20d'] = safe_div(df['PC_Ratio'], pc_20d)
    new_features.append('pc_ratio_rel_20d')

if 'gex_norm' in df.columns:
    # Gamma exposure shift (dealer hedging pressure change)
    df['gex_norm_chg_5d'] = grp_diff('gex_norm', 5)
    new_features.append('gex_norm_chg_5d')

if 'total_oi_norm' in df.columns:
    # Open interest dynamics
    df['total_oi_norm_chg_5d'] = grp_diff('total_oi_norm', 5)
    new_features.append('total_oi_norm_chg_5d')

opt_count = sum(1 for f in ['pc_ratio_chg_5d','pc_ratio_rel_20d','gex_norm_chg_5d','total_oi_norm_chg_5d'] if f in df.columns)
print(f"  C8 Options positioning dynamics: {opt_count} features")

# ── C9. PRICE IMPACT DYNAMICS ───────────────────────────────────────────────

if 'percentpriceimpact_lr_ave' in df.columns:
    pi_5d = grp_roll('percentpriceimpact_lr_ave', 5)
    df['priceimpact_rel_5d'] = safe_div(df['percentpriceimpact_lr_ave'], pi_5d)
    new_features.append('priceimpact_rel_5d')

if 'percentrealizedspread_lr_ave' in df.columns:
    rs_5d = grp_roll('percentrealizedspread_lr_ave', 5)
    df['realspread_rel_5d'] = safe_div(df['percentrealizedspread_lr_ave'], rs_5d)
    new_features.append('realspread_rel_5d')

pi_count = sum(1 for f in ['priceimpact_rel_5d','realspread_rel_5d'] if f in df.columns)
print(f"  C9 Price impact dynamics: {pi_count} features")

# ── C10. DEPTH DYNAMICS ─────────────────────────────────────────────────────

# Depth imbalance dynamics (already computed depth_imbalance in Section A)
df['depth_imbalance_5d_mean'] = grp_roll('depth_imbalance', 5)
new_features.append('depth_imbalance_5d_mean')

df['depth_imbalance_chg_1d'] = grp_diff('depth_imbalance', 1)
new_features.append('depth_imbalance_chg_1d')

# Bid depth withdrawal signal: today's bid depth / 5-day average
if 'bestbiddepth_share_tw_to_shrout' in df.columns:
    bid_d_5d = grp_roll('bestbiddepth_share_tw_to_shrout', 5)
    df['bid_depth_rel_5d'] = safe_div(df['bestbiddepth_share_tw_to_shrout'], bid_d_5d)
    new_features.append('bid_depth_rel_5d')

if 'bestofrdepth_share_tw_to_shrout' in df.columns:
    ofr_d_5d = grp_roll('bestofrdepth_share_tw_to_shrout', 5)
    df['offer_depth_rel_5d'] = safe_div(df['bestofrdepth_share_tw_to_shrout'], ofr_d_5d)
    new_features.append('offer_depth_rel_5d')

depth_count = sum(1 for f in ['depth_imbalance_5d_mean','depth_imbalance_chg_1d','bid_depth_rel_5d','offer_depth_rel_5d'] if f in df.columns)
print(f"  C10 Depth dynamics: {depth_count} features")

# ── C11. CROSS-METRIC INTERACTIONS ──────────────────────────────────────────

# 20-day rolling correlation between return and turnover
# Positive = high volume on up days (conviction), negative = selling pressure
df['volume_return_corr_20d'] = df.groupby('permno').apply(
    lambda g: g['dlyretx'].rolling(20, min_periods=10).corr(g['turnover'])
).reset_index(level=0, drop=True)
new_features.append('volume_return_corr_20d')

print(f"  C11 Cross-metric interactions: 1 feature")

c_total = len(new_features) - c_start
print(f"\n  Section C total: {c_total} rolling/dynamic features")

# ═══════════════════════════════════════════════════════════════════════════════
# D. DROP ORIGINAL RAW COLUMNS REPLACED BY DERIVED/NORMALISED VERSIONS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n--- D. Drop original raw columns ---\n")

# All price-level columns (replaced by Section A derived ratios)
price_level_cols = [
    # CRSP prices
    'dlyprc', 'dlyopen', 'dlyhigh', 'dlylow', 'dlybid', 'dlyask',
    # TAQ average prices
    'avg_buy_price_inst50k', 'avg_buy_price_lr', 'avg_buy_price_retail',
    'avg_price_a', 'avg_price_b', 'avg_price_m',
    'avg_sell_price_inst50k', 'avg_sell_price_lr', 'avg_sell_price_retail',
    # TAQ midpoints
    'mid_1pm', 'mid_4pm', 'mid_after_open', 'mid_before_close',
    # TAQ NBBO
    'nbb_1pm', 'nbb_4pm', 'nbb_after_open', 'nbb_before_close',
    'nbo_1pm', 'nbo_4pm', 'nbo_after_open', 'nbo_before_close',
    # TAQ pre-trade midpoints
    'ptime_1pm', 'ptime_4pm', 'ptime_close', 'ptime_open',
    # TAQ close/open prices
    'cprc', 'oprc',
    # TAQ VWAP by venue
    'vw_price_a', 'vw_price_b', 'vw_price_m',
    # TAQ VWAP by direction
    'vwavg_buy_price_lr', 'vwavg_sell_price_lr',
    # TAQ price extremes
    'price_high_a', 'price_high_b', 'price_high_m',
    'price_low_a', 'price_low_b', 'price_low_m',
]

# All dollar volume originals (replaced by _to_cap versions)
# All share volume originals (replaced by _to_shrout versions)
# All count originals (replaced by _pct versions)
# Plus helper and replaced columns

raw_to_drop = (
    price_level_cols +
    dollar_vol_cols +
    share_vol_cols +
    count_cols +
    ['dlyprcvol', 'dlyvol',       # replaced by dvol_to_cap and turnover
     'shrout',                     # helper only (normalisation denominator)
     'total_trade',                # helper only (count normalisation denominator)
     'stime_close', 'stime_open', # timing in seconds, not useful cross-sectionally
    ]
)

# Only drop columns that actually exist
drops_present = [c for c in raw_to_drop if c in df.columns]
drops_missing = [c for c in raw_to_drop if c not in df.columns]

df = df.drop(columns=drops_present)

print(f"  Dropped {len(drops_present)} raw/helper columns")
if drops_missing:
    print(f"  ({len(drops_missing)} in drop list not found — already removed)")

# ═══════════════════════════════════════════════════════════════════════════════
# SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("BLOCK 3: SUMMARY")
print("=" * 90)

remaining = [c for c in df.columns if c not in ['permno', 'date', 'dlyret', 'dlycap']]
remaining_numeric = [c for c in remaining if pd.api.types.is_numeric_dtype(df[c])]

print(f"\n  New features by section:")
print(f"    A. Price-derived ratios:        {len([f for f in new_features if not f.endswith(('_to_cap','_to_shrout','_pct')) and new_features.index(f) < b_start])}")
print(f"    B. Normalised vol/count:        {len([f for f in new_features if f.endswith(('_to_cap','_to_shrout','_pct'))])}")
print(f"    C. Rolling/dynamic:             {c_total}")
print(f"    ──────────────────────────────")
print(f"    Total new features:             {len(new_features)}")
print(f"\n  Raw columns dropped:              {len(drops_present)}")
print(f"  Net column change:                {n_before} → {df.shape[1]} columns")
print(f"  Surviving factors:                {len(remaining)} ({len(remaining_numeric)} numeric)")

# Verify no required columns were dropped
for c in ['permno', 'date', 'dlyret', 'dlycap']:
    assert c in df.columns, f"FATAL: Required column '{c}' was dropped!"
print(f"\n  ✓ Required columns intact")

# Verify all new features exist
missing_new = [f for f in new_features if f not in df.columns]
if missing_new:
    print(f"  ✗ Missing new features: {missing_new}")
else:
    print(f"  ✓ All {len(new_features)} new features present")

# Verify no price-level columns remain
remaining_prices = [c for c in df.columns if c in price_level_cols]
if remaining_prices:
    print(f"  ⚠ Price-level columns still present: {remaining_prices}")
else:
    print(f"  ✓ All price-level columns removed — every factor is stock-comparable")

# NaN check on rolling features (expected: early rows will be NaN due to warmup)
print(f"\n  Rolling feature NaN rates (includes warmup periods):")
rolling_features = [f for f in new_features if any(x in f for x in
    ['_5d', '_20d', '_rel_', '_chg_', '_corr_', '_mean'])]
for f in rolling_features[:15]:
    nan_pct = df[f].isna().mean() * 100
    print(f"    {f:<35s} {nan_pct:>5.2f}%")
if len(rolling_features) > 15:
    print(f"    ... and {len(rolling_features) - 15} more")

print(f"\n  Complete list of {len(new_features)} new features:")
for i, f in enumerate(new_features, 1):
    print(f"    {i:>3d}. {f}")

BLOCK 3: FEATURE ENGINEERING

--- A. Derived ratio features from price-level pairs ---

  Section A: 21 price-derived features created

--- B. Normalise volume, dollar volume, and count columns ---

  Dollar volume → /dlycap: 15 normalised
  Share volume → /shrout: 27 normalised
  Trade counts → /total_trade: 13 normalised
  Section B total: 55 normalised features

--- C. Rolling & dynamic features (per-stock) ---

  Computing rolling features (this may take a few minutes)...

  C1 Return momentum: 4 features
  C2 Return volatility: 3 features
  C3 Volume dynamics: 5 features
  C4 Liquidity dynamics: 5 features
  C5 Intraday dynamics: 3 features
  C6 IV dynamics: 6 features
  C7 Order flow dynamics: 5 features
  C8 Options positioning dynamics: 4 features
  C9 Price impact dynamics: 2 features
  C10 Depth dynamics: 4 features


C:\Users\Henry\AppData\Local\Temp\ipykernel_33956\2015737958.py:457: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df['volume_return_corr_20d'] = df.groupby('permno').apply(


  C11 Cross-metric interactions: 1 feature

  Section C total: 42 rolling/dynamic features

--- D. Drop original raw columns ---

  Dropped 105 raw/helper columns

BLOCK 3: SUMMARY

  New features by section:
    A. Price-derived ratios:        20
    B. Normalised vol/count:        56
    C. Rolling/dynamic:             42
    ──────────────────────────────
    Total new features:             118

  Raw columns dropped:              105
  Net column change:                183 → 196 columns
  Surviving factors:                192 (192 numeric)

  ✓ Required columns intact
  ✓ All 118 new features present
  ✓ All price-level columns removed — every factor is stock-comparable

  Rolling feature NaN rates (includes warmup periods):
    ret_cum_5d                           0.04%
    ret_cum_20d                          0.39%
    ret_max_5d                           0.04%
    ret_min_5d                           0.04%
    ret_vol_5d                           0.04%
    ret_vol_20d           

In [20]:
# Retail dollar volume share (% of total)
if all(c in df.columns for c in ['buy_dv_retail_to_cap', 'sell_dv_retail_to_cap', 'total_dv_lr_to_cap']):
    df['retail_dv_share'] = safe_div(
        df['buy_dv_retail_to_cap'] + df['sell_dv_retail_to_cap'],
        df['total_dv_lr_to_cap']
    )
    print(f"retail_dv_share: median={df['retail_dv_share'].median():.4f}")

retail_dv_share: median=0.0624


In [21]:
# Drop the perfectly redundant feature
df = df.drop(columns=['dvol_to_cap_rel_5d'])
print(f"Dropped dvol_to_cap_rel_5d (r=1.0 with turnover_rel_5d)")

# Add retail dollar volume share
df['retail_dv_share'] = safe_div(
    df['buy_dv_retail_to_cap'] + df['sell_dv_retail_to_cap'],
    df['total_dv_lr_to_cap']
)
print(f"Added retail_dv_share: median={df['retail_dv_share'].median():.4f}")

# Final count
remaining = [c for c in df.columns if c not in ['permno', 'date', 'dlyret', 'dlycap']]
print(f"\nFinal factor count: {len(remaining)}")

Dropped dvol_to_cap_rel_5d (r=1.0 with turnover_rel_5d)
Added retail_dv_share: median=0.0624

Final factor count: 192


In [22]:
# %% [markdown]
# ## Block 4: Final Factor Inventory & Save
#
# Catalogues every surviving factor with description, source, and category.
# Saves the inventory CSV and the engineered panel parquet.

# %%
print("=" * 90)
print("BLOCK 4: FINAL FACTOR INVENTORY & SAVE")
print("=" * 90)

from pathlib import Path

OUT_DIR = Path('../../../Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# BUILD FINAL INVENTORY
# ═══════════════════════════════════════════════════════════════════════════════

# %%
final_inventory = []

def add(col, source, category, description):
    final_inventory.append({
        'column': col,
        'source': source,
        'category': category,
        'description': description,
    })

# ─────────────────────────────────────────────────────────────────────────────
# CRSP-DERIVED (4 factors)
# ─────────────────────────────────────────────────────────────────────────────

add('dlyretx',            'CRSP', 'return',      'Daily return excluding dividends (price return, close-to-close)')
add('dlyreti',            'CRSP', 'return',      'Daily return from dividends only (income return)')
add('bid_ask_spread',     'CRSP', 'liquidity',   'Normalised bid-ask spread: |ask - bid| / midpoint')
add('intraday_range',     'CRSP', 'volatility',  'Normalised daily high-low range: (high - low) / close')
add('open_to_close_ret',  'CRSP', 'return',      'Intraday open-to-close return: (close - open) / open')
add('turnover',           'CRSP', 'volume',      'Daily share turnover: volume / shares outstanding')
add('dvol_to_cap',        'CRSP', 'volume',      'Dollar volume intensity: dollar volume / market cap')

# ─────────────────────────────────────────────────────────────────────────────
# TAQ: SPREADS & PRICE IMPACT (already ratios, kept as-is from original)
# ─────────────────────────────────────────────────────────────────────────────

add('quotedspread_dollar_tw',       'TAQ', 'liquidity',     'Time-weighted quoted spread in dollars')
add('quotedspread_percent_tw',      'TAQ', 'liquidity',     'Time-weighted quoted spread as % of midpoint')
add('effectivespread_dollar_ave',   'TAQ', 'liquidity',     'Effective spread in dollars, equal-weighted avg')
add('effectivespread_dollar_dw',    'TAQ', 'liquidity',     'Effective spread in dollars, dollar-weighted')
add('effectivespread_dollar_sw',    'TAQ', 'liquidity',     'Effective spread in dollars, share-weighted')
add('effectivespread_percent_ave',  'TAQ', 'liquidity',     'Effective spread as %, equal-weighted avg')
add('effectivespread_percent_dw',   'TAQ', 'liquidity',     'Effective spread as %, dollar-weighted')
add('effectivespread_percent_sw',   'TAQ', 'liquidity',     'Effective spread as %, share-weighted')
add('dollarpriceimpact_lr_ave',     'TAQ', 'price_impact',  'Dollar price impact, Lee-Ready, equal-weighted avg')
add('dollarpriceimpact_lr_dw',      'TAQ', 'price_impact',  'Dollar price impact, Lee-Ready, dollar-weighted')
add('dollarpriceimpact_lr_sw',      'TAQ', 'price_impact',  'Dollar price impact, Lee-Ready, share-weighted')
add('dollarrealizedspread_lr_ave',  'TAQ', 'price_impact',  'Dollar realized spread, Lee-Ready, equal-weighted avg')
add('dollarrealizedspread_lr_dw',   'TAQ', 'price_impact',  'Dollar realized spread, Lee-Ready, dollar-weighted')
add('dollarrealizedspread_lr_sw',   'TAQ', 'price_impact',  'Dollar realized spread, Lee-Ready, share-weighted')
add('percentpriceimpact_lr_ave',    'TAQ', 'price_impact',  'Percent price impact, Lee-Ready, equal-weighted avg')
add('percentpriceimpact_lr_dw',     'TAQ', 'price_impact',  'Percent price impact, Lee-Ready, dollar-weighted')
add('percentpriceimpact_lr_sw',     'TAQ', 'price_impact',  'Percent price impact, Lee-Ready, share-weighted')
add('percentrealizedspread_lr_ave', 'TAQ', 'price_impact',  'Percent realized spread, Lee-Ready, equal-weighted avg')
add('percentrealizedspread_lr_dw',  'TAQ', 'price_impact',  'Percent realized spread, Lee-Ready, dollar-weighted')
add('percentrealizedspread_lr_sw',  'TAQ', 'price_impact',  'Percent realized spread, Lee-Ready, share-weighted')

# ─────────────────────────────────────────────────────────────────────────────
# TAQ: BUY-SELL RATIOS (already bounded 0-1, kept as-is)
# ─────────────────────────────────────────────────────────────────────────────

add('bs_ratio_inst50k_num',  'TAQ', 'order_flow', 'Buy-sell ratio by trade count, institutional >$50K')
add('bs_ratio_inst50k_vol',  'TAQ', 'order_flow', 'Buy-sell ratio by volume, institutional >$50K')
add('bs_ratio_num',          'TAQ', 'order_flow', 'Buy-sell ratio by trade count, all trades')
add('bs_ratio_retail_num',   'TAQ', 'order_flow', 'Buy-sell ratio by trade count, retail')
add('bs_ratio_retail_vol',   'TAQ', 'order_flow', 'Buy-sell ratio by volume, retail')
add('bs_ratio_vol',          'TAQ', 'order_flow', 'Buy-sell ratio by volume, all trades')

# ─────────────────────────────────────────────────────────────────────────────
# TAQ: MICROSTRUCTURE QUALITY (already ratios, kept as-is)
# ─────────────────────────────────────────────────────────────────────────────

add('hindex',          'TAQ', 'concentration', 'Herfindahl index of trade size concentration')
add('ivol_q',          'TAQ', 'volatility',    'Intraday quote-midpoint volatility')
add('ivol_t',          'TAQ', 'volatility',    'Intraday trade-price volatility')
add('n30_pos',         'TAQ', 'momentum',      'Number of 30-sec intervals with positive midpoint returns')
add('n5_pos',          'TAQ', 'momentum',      'Number of 5-min intervals with positive midpoint returns')
add('n_obs',           'TAQ', 'coverage',      'Number of observation intervals with valid quotes')
add('ret_mkt_m',       'TAQ', 'return',        'Intraday return from midpoint prices (open to close)')
add('tsignsqrtdvol1',  'TAQ', 'price_impact',  'Trade sign × sqrt(dollar volume), metric 1 (Kyle lambda)')
add('tsignsqrtdvol2',  'TAQ', 'price_impact',  'Trade sign × sqrt(dollar volume), metric 2 (Kyle lambda)')
add('var_ratio1',      'TAQ', 'efficiency',    'Variance ratio at horizon 1')
add('var_ratio2',      'TAQ', 'efficiency',    'Variance ratio at horizon 2')
add('var_ratio3',      'TAQ', 'efficiency',    'Variance ratio at horizon 3')
add('var_ratio4',      'TAQ', 'efficiency',    'Variance ratio at horizon 4')
add('var_ratio5',      'TAQ', 'efficiency',    'Variance ratio at horizon 5')

# ─────────────────────────────────────────────────────────────────────────────
# TAQ: PRICE-DERIVED RATIOS (created in Block 3 Section A)
# ─────────────────────────────────────────────────────────────────────────────

add('buysell_premium_lr',       'TAQ', 'order_flow',  'Buy-sell price premium, Lee-Ready: (avg_buy - avg_sell) / close')
add('buysell_premium_inst50k',  'TAQ', 'order_flow',  'Buy-sell price premium, institutional >$50K')
add('buysell_premium_retail',   'TAQ', 'order_flow',  'Buy-sell price premium, retail trades')
add('venue_range_m',            'TAQ', 'volatility',  'Normalised intraday range, all exchanges: (high - low) / close')
add('venue_range_a',            'TAQ', 'volatility',  'Normalised intraday range, ARCA exchange')
add('venue_range_b',            'TAQ', 'volatility',  'Normalised intraday range, BATS exchange')
add('intraday_drift',           'TAQ', 'momentum',    'Intraday price drift: (mid_4pm - mid_open) / mid_open')
add('midday_drift',             'TAQ', 'momentum',    'Midday price drift: (mid_4pm - mid_1pm) / mid_1pm')
add('morning_drift',            'TAQ', 'momentum',    'Morning price drift: (mid_1pm - mid_open) / mid_open')
add('nbbo_spread_1pm',          'TAQ', 'liquidity',   'NBBO spread at 1:00 PM as % of midpoint')
add('nbbo_spread_4pm',          'TAQ', 'liquidity',   'NBBO spread at 4:00 PM as % of midpoint')
add('nbbo_spread_open',         'TAQ', 'liquidity',   'NBBO spread shortly after open as % of midpoint')
add('nbbo_spread_close',        'TAQ', 'liquidity',   'NBBO spread shortly before close as % of midpoint')
add('close_vs_mid',             'TAQ', 'order_flow',  'Closing trade vs midpoint: (close - mid_4pm) / mid_4pm')
add('open_vs_mid',              'TAQ', 'order_flow',  'Opening trade vs midpoint: (open - mid_open) / mid_open')
add('vwap_deviation_m',         'TAQ', 'execution',   'VWAP deviation from close: (VWAP - close) / close')
add('depth_imbalance',          'TAQ', 'depth',       'Book imbalance: (bid_depth - offer_depth) / (bid + offer)')

# ─────────────────────────────────────────────────────────────────────────────
# TAQ: NORMALISED DOLLAR VOLUMES (÷ market cap, created in Block 3 Section B)
# ─────────────────────────────────────────────────────────────────────────────

add('buy_dv_inst50k_to_cap',       'TAQ', 'order_flow', 'Inst >$50K buy dollar volume / market cap')
add('buy_dv_lr_to_cap',            'TAQ', 'order_flow', 'Lee-Ready buy dollar volume / market cap')
add('buy_dv_retail_to_cap',        'TAQ', 'order_flow', 'Retail buy dollar volume / market cap')
add('sell_dv_inst50k_to_cap',      'TAQ', 'order_flow', 'Inst >$50K sell dollar volume / market cap')
add('sell_dv_lr_to_cap',           'TAQ', 'order_flow', 'Lee-Ready sell dollar volume / market cap')
add('sell_dv_retail_to_cap',       'TAQ', 'order_flow', 'Retail sell dollar volume / market cap')
add('total_dv_inst50k_to_cap',     'TAQ', 'order_flow', 'Inst >$50K total dollar volume / market cap')
add('total_dv_lr_to_cap',          'TAQ', 'order_flow', 'Lee-Ready total dollar volume / market cap')
add('total_dv_retail_to_cap',      'TAQ', 'order_flow', 'Retail total dollar volume / market cap')
add('total_dollar_a_to_cap',       'TAQ', 'venue',      'ARCA dollar volume / market cap')
add('total_dollar_b_to_cap',       'TAQ', 'venue',      'BATS dollar volume / market cap')
add('total_dollar_m_to_cap',       'TAQ', 'venue',      'All-exchange dollar volume / market cap')
add('iso_dollar_to_cap',           'TAQ', 'order_flow', 'ISO dollar volume / market cap')
add('bestbiddepth_dollar_tw_to_cap','TAQ','depth',      'TW best bid depth (dollars) / market cap')
add('bestofrdepth_dollar_tw_to_cap','TAQ','depth',      'TW best offer depth (dollars) / market cap')

# ─────────────────────────────────────────────────────────────────────────────
# TAQ: NORMALISED SHARE VOLUMES (÷ shares outstanding, Block 3 Section B)
# ─────────────────────────────────────────────────────────────────────────────

add('buyvol_inst50k_to_shrout',     'TAQ', 'order_flow', 'Inst >$50K buy volume / shares outstanding')
add('buyvol_lr_to_shrout',          'TAQ', 'order_flow', 'Lee-Ready buy volume / shares outstanding')
add('buyvol_retail_to_shrout',      'TAQ', 'order_flow', 'Retail buy volume / shares outstanding')
add('sellvol_inst50k_to_shrout',    'TAQ', 'order_flow', 'Inst >$50K sell volume / shares outstanding')
add('sellvol_lr_to_shrout',         'TAQ', 'order_flow', 'Lee-Ready sell volume / shares outstanding')
add('sellvol_retail_to_shrout',     'TAQ', 'order_flow', 'Retail sell volume / shares outstanding')
add('total_vol_to_shrout',          'TAQ', 'volume',     'Total volume / shares outstanding (turnover, TAQ)')
add('total_vol_a_to_shrout',        'TAQ', 'venue',      'ARCA volume / shares outstanding')
add('total_vol_b_to_shrout',        'TAQ', 'venue',      'BATS volume / shares outstanding')
add('total_vol_inst50k_to_shrout',  'TAQ', 'order_flow', 'Inst >$50K volume / shares outstanding')
add('total_vol_m_to_shrout',        'TAQ', 'venue',      'All-exchange volume / shares outstanding (TAQ)')
add('total_vol_retail_to_shrout',   'TAQ', 'order_flow', 'Retail volume / shares outstanding')
add('iso_vol_to_shrout',            'TAQ', 'order_flow', 'ISO volume / shares outstanding')
add('bestbiddepth_share_tw_to_shrout','TAQ','depth',     'TW best bid depth (shares) / shares outstanding')
add('bestofrdepth_share_tw_to_shrout','TAQ','depth',     'TW best offer depth (shares) / shares outstanding')
add('nbbqty_1pm_to_shrout',         'TAQ', 'depth',      'NBB quantity at 1PM / shares outstanding')
add('nbbqty_4pm_to_shrout',         'TAQ', 'depth',      'NBB quantity at 4PM / shares outstanding')
add('nbbqty_after_open_to_shrout',  'TAQ', 'depth',      'NBB quantity after open / shares outstanding')
add('nbbqty_before_close_to_shrout','TAQ', 'depth',      'NBB quantity before close / shares outstanding')
add('nboqty_1pm_to_shrout',         'TAQ', 'depth',      'NBO quantity at 1PM / shares outstanding')
add('nboqty_4pm_to_shrout',         'TAQ', 'depth',      'NBO quantity at 4PM / shares outstanding')
add('nboqty_after_open_to_shrout',  'TAQ', 'depth',      'NBO quantity after open / shares outstanding')
add('nboqty_before_close_to_shrout','TAQ', 'depth',      'NBO quantity before close / shares outstanding')
add('csize_to_shrout',              'TAQ', 'trade_size', 'Closing trade size / shares outstanding')
add('osize_to_shrout',              'TAQ', 'trade_size', 'Opening trade size / shares outstanding')
add('size_1pm_to_shrout',           'TAQ', 'trade_size', 'Trade size at 1PM / shares outstanding')
add('size_4pm_to_shrout',           'TAQ', 'trade_size', 'Trade size at 4PM / shares outstanding')

# ─────────────────────────────────────────────────────────────────────────────
# TAQ: NORMALISED TRADE COUNTS (÷ total trades, Block 3 Section B)
# ─────────────────────────────────────────────────────────────────────────────

add('buynumtrades_inst50k_pct',    'TAQ', 'order_flow', '% of trades that are inst >$50K buys')
add('buynumtrades_lr_pct',         'TAQ', 'order_flow', '% of trades that are Lee-Ready buys')
add('buynumtrades_retail_pct',     'TAQ', 'order_flow', '% of trades that are retail buys')
add('sellnumtrades_inst50k_pct',   'TAQ', 'order_flow', '% of trades that are inst >$50K sells')
add('sellnumtrades_lr_pct',        'TAQ', 'order_flow', '% of trades that are Lee-Ready sells')
add('sellnumtrades_retail_pct',    'TAQ', 'order_flow', '% of trades that are retail sells')
add('total_trade_inst50k_pct',     'TAQ', 'order_flow', '% of trades that are institutional >$50K')
add('total_trade_retail_pct',      'TAQ', 'order_flow', '% of trades that are retail')
add('total_n_trades_a_pct',        'TAQ', 'venue',      '% of trades on ARCA exchange')
add('total_n_trades_b_pct',        'TAQ', 'venue',      '% of trades on BATS exchange')
add('total_n_trades_m_pct',        'TAQ', 'venue',      '% of trades on all exchanges (TAQ)')
add('n_iso_trade_pct',             'TAQ', 'order_flow', '% of trades that are intermarket sweeps')
add('n_outside_nbbo_trade_pct',    'TAQ', 'order_flow', '% of trades executed outside NBBO')

# ─────────────────────────────────────────────────────────────────────────────
# TAQ + CRSP: ROLLING RETURN FEATURES (Block 3 Section C)
# ─────────────────────────────────────────────────────────────────────────────

add('ret_cum_5d',      'CRSP', 'momentum',   'Cumulative return over last 5 trading days')
add('ret_cum_20d',     'CRSP', 'momentum',   'Cumulative return over last 20 trading days')
add('ret_max_5d',      'CRSP', 'tail_risk',  'Maximum single-day return in last 5 days')
add('ret_min_5d',      'CRSP', 'tail_risk',  'Minimum single-day return in last 5 days')
add('ret_vol_5d',      'CRSP', 'volatility', 'Rolling 5-day return std deviation')
add('ret_vol_20d',     'CRSP', 'volatility', 'Rolling 20-day return std deviation')
add('ret_vol_ratio',   'CRSP', 'volatility', 'Vol regime: 5d vol / 20d vol. >1 = vol spiking.')

# ─────────────────────────────────────────────────────────────────────────────
# ROLLING VOLUME DYNAMICS (Block 3 Section C)
# ─────────────────────────────────────────────────────────────────────────────

add('turnover_5d_mean',   'CRSP', 'volume', 'Smoothed 5-day average turnover')
add('turnover_20d_mean',  'CRSP', 'volume', 'Smoothed 20-day average turnover')
add('turnover_rel_5d',    'CRSP', 'volume', 'Volume surprise: today turnover / 5d avg. >1 = unusual.')
add('turnover_rel_20d',   'CRSP', 'volume', 'Volume surprise: today turnover / 20d avg. >1 = unusual.')

# ─────────────────────────────────────────────────────────────────────────────
# ROLLING LIQUIDITY DYNAMICS (Block 3 Section C)
# ─────────────────────────────────────────────────────────────────────────────

add('spread_5d_mean',        'CRSP', 'liquidity', 'Smoothed 5-day avg bid-ask spread')
add('spread_20d_mean',       'CRSP', 'liquidity', 'Smoothed 20-day avg bid-ask spread')
add('spread_rel_5d',         'CRSP', 'liquidity', 'Liquidity shock: today spread / 5d avg. >1 = deteriorating.')
add('spread_rel_20d',        'CRSP', 'liquidity', 'Liquidity shock: today spread / 20d avg. >1 = deteriorating.')
add('effspread_pct_rel_5d',  'TAQ',  'liquidity', 'Effective spread surprise: today / 5d avg')

# ─────────────────────────────────────────────────────────────────────────────
# ROLLING INTRADAY DYNAMICS (Block 3 Section C)
# ─────────────────────────────────────────────────────────────────────────────

add('intraday_range_rel_5d',     'CRSP', 'volatility', 'Range breakout: today range / 5d avg. >1 = expanding.')
add('intraday_range_5d_mean',    'CRSP', 'volatility', 'Smoothed 5-day average intraday range')
add('open_to_close_ret_5d_mean', 'CRSP', 'momentum',   'Smoothed 5-day avg intraday direction (persistent drift)')

# ─────────────────────────────────────────────────────────────────────────────
# ROLLING IV DYNAMICS (Block 3 Section C)
# ─────────────────────────────────────────────────────────────────────────────

add('iv_catm_chg_1d',    'OM', 'volatility',   '1-day change in ATM call implied vol')
add('iv_catm_chg_5d',    'OM', 'volatility',   '5-day change in ATM call implied vol')
add('iv_catm_rel_20d',   'OM', 'volatility',   'IV regime: current IV / 20d avg. >1 = elevated.')
add('vrp_rv_chg_5d',     'OM', 'volatility',   '5-day change in vol risk premium (IV - RV)')
add('skew_chg_5d',       'OM', 'vol_surface',  '5-day change in OTM skew (tail fear shift)')
add('vol_term_chg_5d',   'OM', 'vol_surface',  '5-day change in vol term structure')

# ─────────────────────────────────────────────────────────────────────────────
# ROLLING ORDER FLOW DYNAMICS (Block 3 Section C)
# ─────────────────────────────────────────────────────────────────────────────

add('bs_ratio_vol_5d_mean',  'TAQ', 'order_flow', 'Smoothed 5-day avg buy-sell ratio (sustained flow)')
add('bs_ratio_vol_chg_1d',   'TAQ', 'order_flow', '1-day change in buy-sell ratio (flow reversal)')
add('bs_ratio_inst_5d_mean', 'TAQ', 'order_flow', 'Smoothed 5-day avg institutional buy-sell ratio')
add('bs_ratio_inst_chg_5d',  'TAQ', 'order_flow', '5-day change in institutional buy-sell ratio')
add('retail_vol_5d_mean',    'TAQ', 'order_flow', 'Smoothed 5-day avg retail volume (participation trend)')

# ─────────────────────────────────────────────────────────────────────────────
# ROLLING OPTIONS POSITIONING DYNAMICS (Block 3 Section C)
# ─────────────────────────────────────────────────────────────────────────────

add('pc_ratio_chg_5d',       'OM', 'sentiment',    '5-day change in put-call ratio (sentiment shift)')
add('pc_ratio_rel_20d',      'OM', 'sentiment',    'Put-call ratio regime: current / 20d avg')
add('gex_norm_chg_5d',       'OM', 'positioning',  '5-day change in gamma exposure (dealer hedging shift)')
add('total_oi_norm_chg_5d',  'OM', 'positioning',  '5-day change in total OI (options market growth/decline)')

# ─────────────────────────────────────────────────────────────────────────────
# ROLLING PRICE IMPACT DYNAMICS (Block 3 Section C)
# ─────────────────────────────────────────────────────────────────────────────

add('priceimpact_rel_5d',   'TAQ', 'price_impact', 'Price impact surprise: today / 5d avg')
add('realspread_rel_5d',    'TAQ', 'price_impact', 'Realized spread surprise: today / 5d avg')

# ─────────────────────────────────────────────────────────────────────────────
# ROLLING DEPTH DYNAMICS (Block 3 Section C)
# ─────────────────────────────────────────────────────────────────────────────

add('depth_imbalance_5d_mean', 'TAQ', 'depth', 'Smoothed 5-day avg book imbalance')
add('depth_imbalance_chg_1d',  'TAQ', 'depth', '1-day change in book imbalance')
add('bid_depth_rel_5d',        'TAQ', 'depth', 'Bid depth surprise: today / 5d avg. <1 = depth withdrawal.')
add('offer_depth_rel_5d',      'TAQ', 'depth', 'Offer depth surprise: today / 5d avg. <1 = depth withdrawal.')

# ─────────────────────────────────────────────────────────────────────────────
# CROSS-METRIC INTERACTION (Block 3 Section C)
# ─────────────────────────────────────────────────────────────────────────────

add('volume_return_corr_20d', 'CRSP+TAQ', 'cross_metric', 'Rolling 20d corr(return, turnover). +ve = conviction, -ve = panic selling.')

# ─────────────────────────────────────────────────────────────────────────────
# RETAIL FLOW (added post Block 3)
# ─────────────────────────────────────────────────────────────────────────────

add('retail_dv_share', 'TAQ', 'order_flow', 'Retail dollar volume as % of total LR dollar volume')

# ─────────────────────────────────────────────────────────────────────────────
# OPTIONMETRICS: IMPLIED VOLATILITY & SURFACE (kept as-is)
# ─────────────────────────────────────────────────────────────────────────────

add('iv_catm',            'OM', 'volatility',   'Implied volatility, call ATM, ~30d tenor')
add('iv_PATM',            'OM', 'volatility',   'Implied volatility, put ATM, ~30d tenor')
add('iv_POTM',            'OM', 'volatility',   'Implied volatility, put OTM')
add('iv_91d_atm',         'OM', 'volatility',   'Implied volatility, ATM, 91d tenor')
add('iv_30d_call25',      'OM', 'volatility',   'Implied volatility, 30d, 25-delta OTM call')
add('iv_30d_put25',       'OM', 'volatility',   'Implied volatility, 30d, 25-delta OTM put')
add('vol_term_structure',  'OM', 'vol_surface', 'IV term structure: 91d ATM minus 30d ATM')
add('vol_smile',           'OM', 'vol_surface', 'Vol smile: put25 + call25 - 2×ATM')
add('Skew_OTM',           'OM', 'vol_surface',  'OTM skew: IV_POTM minus IV_CATM')

# ─────────────────────────────────────────────────────────────────────────────
# OPTIONMETRICS: REALISED VOL & VRP (kept as-is)
# ─────────────────────────────────────────────────────────────────────────────

add('hvol',     'OM', 'volatility', 'Historical realised volatility, 30d')
add('rv_30d',   'OM', 'volatility', 'Realised volatility, 30d, from daily returns')
add('vrp_rv',   'OM', 'volatility', 'Vol risk premium: IV_CATM minus RV_30d')
add('vrp_hvol', 'OM', 'volatility', 'Vol risk premium: IV_CATM minus hvol')

# ─────────────────────────────────────────────────────────────────────────────
# OPTIONMETRICS: PUT-CALL & PARITY (kept as-is)
# ─────────────────────────────────────────────────────────────────────────────

add('Parity_VSpread', 'OM', 'options',   'Put-call parity violation spread')
add('nopt_Parity',    'OM', 'options',   'Number of option pairs used in parity calc')
add('PC_Ratio',       'OM', 'sentiment', 'Put-call volume ratio. >1 = bearish, <1 = bullish.')

# ─────────────────────────────────────────────────────────────────────────────
# OPTIONMETRICS: GREEKS & POSITIONING (kept as-is, already normalised)
# ─────────────────────────────────────────────────────────────────────────────

add('oi_wt_delta',             'OM', 'positioning', 'OI-weighted avg delta (net directional tilt)')
add('oi_wt_gamma',             'OM', 'positioning', 'OI-weighted avg gamma (convexity exposure)')
add('oi_wt_vega',              'OM', 'positioning', 'OI-weighted avg vega (vol sensitivity)')
add('oi_wt_theta',             'OM', 'positioning', 'OI-weighted avg theta (time decay)')
add('gex_norm',                'OM', 'positioning', 'Gamma exposure normalised by market cap')
add('dex_norm',                'OM', 'positioning', 'Delta exposure normalised by market cap')
add('delta_dollar_volume_norm','OM', 'positioning', 'Delta-weighted dollar volume / market cap')
add('total_oi_norm',           'OM', 'positioning', 'Total open interest / market cap')
add('total_volume_norm',       'OM', 'positioning', 'Total options volume / market cap')

# ─────────────────────────────────────────────────────────────────────────────
# OPTIONMETRICS: MONEYNESS DISTRIBUTION (kept as-is)
# ─────────────────────────────────────────────────────────────────────────────

add('sumOI_c_money1_pct', 'OM', 'positioning', 'Call OI near ATM as % of total OI')
add('sumOI_c_money2_pct', 'OM', 'positioning', 'Call OI moderate OTM as % of total OI')
add('sumOI_c_money3_pct', 'OM', 'positioning', 'Call OI deep OTM as % of total OI')
add('sumOI_p_money1_pct', 'OM', 'positioning', 'Put OI near ATM as % of total OI')
add('sumOI_p_money2_pct', 'OM', 'positioning', 'Put OI moderate OTM as % of total OI')
add('sumOI_p_money3_pct', 'OM', 'positioning', 'Put OI deep OTM as % of total OI')

# ═══════════════════════════════════════════════════════════════════════════════
# VALIDATE INVENTORY VS ACTUAL DATA
# ═══════════════════════════════════════════════════════════════════════════════

# %%
inv = pd.DataFrame(final_inventory)

remaining_factors = [c for c in df.columns if c not in ['permno', 'date', 'dlyret', 'dlycap']]
catalogued = set(inv['column'])

in_data_not_catalogued = [c for c in remaining_factors if c not in catalogued]
in_catalogue_not_data = [c for c in catalogued if c not in remaining_factors]

print(f"\n  Factors in data:       {len(remaining_factors)}")
print(f"  Factors catalogued:    {len(inv)}")

if in_data_not_catalogued:
    print(f"\n  ✗ IN DATA but NOT catalogued ({len(in_data_not_catalogued)}):")
    for c in in_data_not_catalogued:
        print(f"    {c}")
else:
    print(f"  ✓ Every factor in data is catalogued")

if in_catalogue_not_data:
    print(f"\n  ✗ In catalogue but NOT in data ({len(in_catalogue_not_data)}):")
    for c in in_catalogue_not_data:
        print(f"    {c}")
else:
    print(f"  ✓ Every catalogued factor exists in data")

# Summary tables
print(f"\n  By source:")
print(inv['source'].value_counts().to_string())

print(f"\n  By category:")
print(inv['category'].value_counts().to_string())

# ═══════════════════════════════════════════════════════════════════════════════
# SAVE INVENTORY CSV
# ═══════════════════════════════════════════════════════════════════════════════

# %%
csv_path = OUT_DIR / 'stock_daily_factor_inventory_final.csv'
csv_string = inv.to_csv(index=False)
with open(csv_path, 'w', encoding='utf-8') as f:
    f.write(csv_string)
print(f"\n  ✓ Inventory saved: {csv_path}")
print(f"    {len(inv)} factors")

# ═══════════════════════════════════════════════════════════════════════════════
# SAVE ENGINEERED PANEL TO PARQUET
# ═══════════════════════════════════════════════════════════════════════════════

# %%
df = df.sort_values(['permno', 'date']).reset_index(drop=True)

parquet_path = OUT_DIR / 'panel_stock_daily_engineered.parquet'
df.to_parquet(parquet_path, index=False, engine='pyarrow')

file_size = parquet_path.stat().st_size
print(f"\n  ✓ Panel saved: {parquet_path}")
print(f"    {len(df):,} rows × {df.shape[1]} columns")
print(f"    ID: permno, date")
print(f"    Weight: dlycap")
print(f"    Target: dlyret")
print(f"    Factors: {len(remaining_factors)}")
print(f"    Size: {file_size / 1e9:.2f} GB")

# ═══════════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("PANEL A FEATURE ENGINEERING COMPLETE")
print("=" * 90)

print(f"""
  Pipeline: Raw (240 cols) → Block 1 drops (183) → Block 3 engineering (196) → Final ({df.shape[1]})

  Final panel:
    Rows:       {len(df):,}
    Columns:    {df.shape[1]}
    Factors:    {len(remaining_factors)}
    PERMNOs:    {df['permno'].nunique()}
    Dates:      {df['date'].nunique():,}
    Date range: {df['date'].min().date()} → {df['date'].max().date()}

  Saved to:
    Panel:     {parquet_path}
    Inventory: {csv_path}

  Next step: Stage 2 aggregation reads this panel and computes
  cap-weighted cross-sectional statistics (mean, std, skew, kurt, spread)
  per date, producing market-level daily time series.
""")

BLOCK 4: FINAL FACTOR INVENTORY & SAVE

  Factors in data:       192
  Factors catalogued:    192
  ✓ Every factor in data is catalogued
  ✓ Every catalogued factor exists in data

  By source:
source
TAQ         125
OM           41
CRSP         25
CRSP+TAQ      1

  By category:
category
order_flow       46
volatility       25
liquidity        18
positioning      17
depth            17
price_impact     16
venue             9
momentum          8
volume            7
vol_surface       5
efficiency        5
return            4
trade_size        4
sentiment         3
options           2
tail_risk         2
coverage          1
concentration     1
cross_metric      1
execution         1

  ✓ Inventory saved: ..\..\..\Data\Data_Collection\Final\Stage_1_5_Validation_and_Feature_Engineering\stock_daily_factor_inventory_final.csv
    192 factors

  ✓ Panel saved: ..\..\..\Data\Data_Collection\Final\Stage_1_5_Validation_and_Feature_Engineering\panel_stock_daily_engineered.parquet
    525,957 rows